<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_04_feature_engineering/stage_04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_04_feature_engineering**

## **Introducción**

Esta notebook corresponde al stage_04a - Technical Indicators del pipeline neural_profit y tiene como objetivo generar, evaluar y consolidar indicadores técnicos intradía para el índice MNQ, a partir de datos minuto a minuto.

Partiendo del dataset intradía ya etiquetado con objetivos de retorno, se calculan indicadores técnicos de forma independiente por jornada, evitando la mezcla de información entre días. Esto garantiza consistencia temporal y previene leakage en etapas posteriores de modelado.

El proceso incluye la evaluación cuantitativa de los indicadores mediante Information Coefficient (IC), utilizando correlación de Spearman entre cada indicador y los targets de retorno definidos para distintos horizontes. Este análisis permite medir no solo la relación promedio con el target, sino también su estabilidad a lo largo del tiempo.

Como resultado final, se generan datasets consolidados y listos para modelado, que incluyen:

- Variables OHLCV
- Targets de retorno a distintos horizontes
- Indicadores técnicos seleccionados y validados

Estos artefactos serán utilizados en las siguientes etapas del pipeline para selección de features, entrenamiento y evaluación de modelos predictivos.

## **0. Configuración del Entorno**


### 0.1. Clonado de repositorio / Acceso a Drive

In [7]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [8]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [9]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

Librería instalada: technical-analysis


### 0.3. Importación de librerías


In [10]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

### 0.4. Definición de rutas



In [11]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [12]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/processed/mnq_intraday_labeled.parquet"))
IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

In [13]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

### 0.5. Códigos auxiliares para carga de datos y visualización


In [14]:
def load_data():

    # Definir la URL del archivo Parquet en Drive
    data_path = f'{drive_path}/data/processed/mnq_intraday_labeled.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [15]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

  return primer_hora, ultima_hora

In [16]:
# Carga del JSON
with IN_ARTIFACT.open("r") as f:
    target_definition_summary = json.load(f)

In [17]:
def print_summary_console(summary: Dict[str, Any]) -> None:
    """Consola legible (sin depender de pandas display)."""
    print("\n" + "=" * 70)
    print(f"STAGE: {summary.get('stage')}")
    print(f"CREATED_AT_UTC: {summary.get('created_at_utc')}")
    print(f"VERSION: {summary.get('version')}")
    print("-" * 70)

    paths = summary.get("paths", {})
    print("[PATHS]")
    for group in ("inputs", "outputs", "reports"):
        print(f"  {group}:")
        for k, v in paths.get(group, {}).items():
            print(f"    - {k}: {v}")

    params = summary.get("params", {})
    print("\n[PARAMS]")
    for k, v in params.items():
        print(f"  - {k}: {v}")

    metrics = summary.get("metrics", {})
    print("\n[METRICS]")
    for k, v in metrics.items():
        print(f"  - {k}: {v}")

    details = summary.get("details", {})
    gw = details.get("gestation_window", {})
    if gw:
        print("\n[GESTATION WINDOW]")
        print(f"  - start: {gw.get('start_hhmm')}")
        print(f"  - end  : {gw.get('end_hhmm')}")
        print(f"  - top_n: {gw.get('top_n')}")
        print(f"  - minute_of_day_min: {gw.get('minute_of_day_min')}")
        print(f"  - minute_of_day_max: {gw.get('minute_of_day_max')}")

    print("=" * 70 + "\n")

In [18]:
def assign_indicator_family(indicator: str) -> str:
    """
    Asigna una familia económica a cada indicador técnico
    en función de su nombre.
    """
    name = indicator.lower()

    if "ema" in name or name.startswith("price_"):
        return "trend_price"

    if name.startswith("bb_"):
        return "volatility_extension"

    if name.startswith("roc") or name.startswith("momentum"):
        return "momentum"

    if name.startswith("rsi"):
        return "momentum_oscillator"

    if name.startswith("stoch"):
        return "momentum_oscillator"

    if name.startswith("atr"):
        return "volatility"

    if name.startswith("volume_ratio"):
        return "volume"

    if name == "macd":
        return "trend_momentum"

    return "other"

In [19]:
def select_top_by_family(
    ic_df: pd.DataFrame,
    top_n: int = 1
) -> pd.DataFrame:
    """
    Selecciona los mejores indicadores por familia
    según abs_IC_delta.
    """
    df = ic_df.copy()

    # Asignar familia
    df["family"] = df["indicator"].apply(assign_indicator_family)

    # Ordenar por fuerza de señal
    df = df.sort_values("abs_IC_delta", ascending=False)

    # Tomar top N por familia
    df_top = (
        df.groupby("family", as_index=False)
          .head(top_n)
          .reset_index(drop=True)
    )

    return df_top

In [20]:
def load_dataset():
  #Carga de dataset base:
  mnq_intraday_labeled = load_data()
  #info_dataset(mnq_intraday_labeled)
  # Eliminación de filas con NaN
  mnq_intraday_labeled = mnq_intraday_labeled.dropna()
  # Verificación posterior
  info_dataset(mnq_intraday_labeled)
  return mnq_intraday_labeled

In [21]:
mnq_intraday_labeled = load_dataset()

# Copiamos para trabajar de forma segura
df = mnq_intraday_labeled.copy()

# -----------------------------
# 1) Selección de columnas base
# -----------------------------
base_cols = ["date", "minute_of_day", "open", "high", "low", "close", "volume"]

# Targets: todas las columnas que empiezan con 'delta_pts_'
target_cols = [c for c in df.columns if c.startswith("delta_pts_")]

# Subset final
df = df[base_cols + target_cols].copy()

# -----------------------------
# 2) Renombrar delta_pts_h -> delta_h
# -----------------------------
rename_map = {
    c: c.replace("delta_pts_", "delta_")
    for c in target_cols
}

df.rename(columns=rename_map, inplace=True)

# -----------------------------
# 3) Sobrescribimos el dataset
# -----------------------------
mnq_intraday_labeled = df.copy()

mnq_intraday_labeled

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 481
	Hora diaria de inicio 06:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York


,date,minute_of_day,open,high,low,close,volume,delta_60,delta_90
datetime,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,390,8727.75,8728.00,8727.75,8727.75,7,9.00,6.00
2019-12-23 06:31:00-05:00,2019-12-23,391,8727.50,8727.75,8726.50,8726.50,89,9.25,7.50
2019-12-23 06:32:00-05:00,2019-12-23,392,8726.50,8726.50,8725.25,8725.25,34,9.75,8.00
2019-12-23 06:33:00-05:00,2019-12-23,393,8725.50,8726.25,8724.75,8726.00,53,8.50,8.00
2019-12-23 06:34:00-05:00,2019-12-23,394,8726.00,8726.00,8726.00,8726.00,3,8.00,7.75
...,...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,866,21716.50,21722.75,21712.00,21719.50,1036,-90.00,-102.00
2025-06-13 14:27:00-04:00,2025-06-13,867,21719.00,21719.75,21695.75,21698.50,3542,-69.25,-74.75
2025-06-13 14:28:00-04:00,2025-06-13,868,21698.25,21700.25,21670.50,21679.25,5241,-52.25,-57.50


## **1. Relación entre indicadores técnicos, information coefficient y los targets definidos del stage_03**

### **1.1. Indicadores técnicos**

Los indicadores técnicos se construyen a partir de la serie de precios intradía, y en particular sobre la variable de cierre (close). Estas transformaciones matemáticas buscan capturar propiedades dinámicas del mercado tales como tendencia, momentum, reversión, volatilidad y estructura temporal del movimiento de precios.

En el marco actual del proyecto, la variable objetivo (target) ya no se define como un retorno normalizado, sino como un movimiento futuro absoluto en puntos (delta en puntos), medido sobre horizontes temporales discretos (60 y 90 minutos). A partir de estos deltas se definen distintos umbrales económicamente relevantes (base, operativo y cola), que dan lugar a señales de trade y targets binarios asociados. Los valores de delta definidos en el stage_03_target_definition son los siguientes:

In [22]:
print_summary_console(target_definition_summary)


STAGE: stage_03b_target_definition
CREATED_AT_UTC: 2026-01-24T23:30:22.837167+00:00
VERSION: 1.0
----------------------------------------------------------------------
[PATHS]
  inputs:
    - intraday_parquet: data/processed/mnq_intraday.parquet
    - stage03a_summary: reports/stage_03a_target_investigation_summary.json
  outputs:
    - mnq_intraday_labeled: data/processed/mnq_intraday_labeled.parquet
  reports:
    - summary: reports/stage_03b_target_definition_summary.json

[PARAMS]
  - date_col: date
  - close_col: close
  - horizons: [60, 90]
  - drop_na_targets: True
  - top_persist_n: 25

[METRICS]
  - n_rows: 626743
  - n_cols: 15
  - n_days: 1303
  - total_nans: 0
  - h60_n_trade: 160670
  - h60_n_target_op: 73450
  - h60_n_target_tail: 20943
  - h90_n_trade: 180453
  - h90_n_target_op: 86779
  - h90_n_target_tail: 25032

[GESTATION WINDOW]
  - start: 08:21
  - end  : 08:49
  - top_n: 25
  - minute_of_day_min: 501
  - minute_of_day_max: 529



Esto implica que, aunque los indicadores técnicos se calculen directamente sobre el precio, su evaluación no se orienta a explicar la evolución instantánea del close, sino a medir su capacidad para anticipar movimientos futuros de magnitud suficiente, expresados en puntos, dentro de un horizonte temporal determinado.

En consecuencia, el vínculo entre indicadores y targets se establece en términos de poder predictivo sobre la ocurrencia de deltas futuros significativos, es decir, sobre la probabilidad de que el mercado alcance determinados umbrales de movimiento (Δ base, Δ operativo o Δ cola) en el horizonte considerado. El análisis posterior se centra, por tanto, en identificar qué indicadores y configuraciones temporales contienen información relevante para discriminar contextos de no-trade, trade operativo o eventos de cola, coherentes con los targets definidos en el stage_03.

### **1.2. Information Coefficient (IC) versus target `delta_h`**

En el dataset `mnq_intraday_labeled`, cada fila representa una decisión potencial en un instante $t$ del intradía. Para ese instante, el target continuo `delta_h` indica el desplazamiento futuro del precio, medido en puntos, desde $t$ hasta $t+h$, para horizontes fijos (por ejemplo, $h = 60$ o $90$ minutos).

La ventana de gestión no redefine el target ni introduce una nueva variable temporal, sino que delimita el conjunto de instantes $t$ en los cuales el sistema está habilitado a evaluar señales y generar decisiones. En este trabajo, dicha ventana se identifica empíricamente como el período donde se observa mayor persistencia estadística de los factores, y constituye el momento operativo en el que el modelo “observa” el mercado.

Por su parte, la ventana de ejecución (o expansión) no es una entidad explícita del modelo ni del cálculo del Information Coefficient (IC). Esta ventana surge de manera implícita, ya que corresponde al intervalo temporal donde se materializa el resultado futuro de las decisiones tomadas en la ventana de gestión. Para cada instante $t$ dentro de la ventana de gestión, el target `delta_h` refleja el comportamiento del precio en $t+h$, que naturalmente cae en una franja horaria posterior.

Con el objetivo de evaluar la robustez temporal de los indicadores técnicos y evitar conclusiones dependientes de un único tramo horario, el análisis de IC se realiza bajo tres contextos complementarios:

a) Jornada completa: se consideran todos los instantes intradía válidos, con el fin de identificar factores estructurales con capacidad predictiva estable a lo largo del día.

b) Ventana de gestación (08:00-09:00): se restringe el análisis al período previo al inicio del movimiento operativo principal, donde suelen formarse las condiciones iniciales del desplazamiento posterior.

c) Ventana de ejecución o expansión (09:00-10:00): se analiza el período inmediatamente posterior, donde los movimientos tienden a desarrollarse con mayor intensidad y direccionalidad.

Este enfoque permite distinguir entre indicadores con valor explicativo global y aquellos cuya capacidad predictiva es dependiente del contexto horario.

Bajo este esquema, el Information Coefficient (IC) se calcula correlacionando, para cada instante $t$ perteneciente al conjunto temporal considerado:

$X_t$: el valor del indicador técnico, calculado exclusivamente con información disponible hasta $t$.

$Y_{t,h}$: el target continuo `delta_h`, asociado a ese mismo instante $t$ y a un horizonte fijo $h$.

De este modo, el IC mide directamente la capacidad del indicador, evaluado en el momento de decisión, para anticipar la magnitud y dirección del movimiento futuro del precio. La relación se establece fila a fila, sin agregar bloques temporales ni comparar ventanas entre sí, respetando estrictamente la causalidad temporal.

En términos operativos, un IC positivo indica que ciertos estados del mercado, caracterizados por los indicadores técnicos en el instante de evaluación, están sistemáticamente asociados a movimientos futuros más favorables según los criterios definidos en el stage_03. Esto justifica su uso como variables explicativas en el entrenamiento del modelo predictivo.


### **1.3. Alineación entre el punto 8 (stage_03b) y el cálculo del IC (stage_04)**

La construcción de las ventanas operativas desarrollada en el punto 8 del stage_03b establece una separación conceptual fundamental entre:

- una ventana de gestación (predicción), donde se origina la información anticipatoria, y

- una ventana de expansión (ejecución), donde los movimientos alcanzan magnitud económica explotable.

Esta separación no entra en conflicto con la definición de los targets ni con el cálculo del Information Coefficient (IC); por el contrario, ambos enfoques son complementarios y coherentes, siempre que se entienda correctamente el rol temporal de cada elemento.



#### **1.3.1. Nivel estadístico (dataset y targets)**


En `mnq_intraday_labeled`, los targets (`delta_pts_h`, `trade_h`, `target_op_h`, `target_tail_h`) están definidos fila a fila, para cada instante t, como el resultado del movimiento futuro observado en t+h.

Esto significa que:
- el target está anclado temporalmente al instante t
- el horizonte h determina cuándo se materializa el resultado,
- no existen targets definidos “por ventana”, sino por decisión potencial en un minuto específico.

Cuando el análisis se restringe a la ventana de gestación (por ejemplo, 08:20-08:40), lo que se hace es seleccionar el subconjunto de instantes
t que, según el análisis empírico del punto 8, concentran información anticipatoria relevante.

En este contexto, el IC mide:
 - La relación estadística entre el estado del mercado en t (capturado por los indicadores técnicos) y el resultado futuro asociado a ese mismo t, que se materializa en la ventana de expansión.

#### **1.3.2. Nivel operativo (modelo y ejecución)**

Desde el punto de vista operativo, el esquema temporal se interpreta de la siguiente manera:

- Ventana de gestación (08:20-08:40)
  - Se calculan los indicadores técnicos.
  - El modelo evalúa si, desde esos instantes, es probable alcanzar un delta relevante a 60 o 90 minutos.
  - Aquí reside la capacidad predictiva.

- Ventana de expansión (09:10-09:40)
  - Es el período donde, empíricamente, los movimientos alcanzan mayor frecuencia y magnitud.
  - Las predicciones generadas previamente habilitan (o no) la toma de operaciones reales.
  - El punto de entrada puede ubicarse en cualquier minuto de esta franja, sujeto a reglas operativas adicionales.

- Horizonte de resultado
- El cierre de la operación ocurre según:
- `delta_op` (objetivo operativo),
- `delta_tail` (extensión),
- o reglas de stop,
  
  siempre respetando el horizonte temporal definido desde el instante de entrada.

#### **1.3.3. Rol del IC dentro de este esquema**

El **Information Coefficient** no evalúa la ejecución, sino la calidad de la información generada en la ventana de gestación.

Su función es responder a la pregunta:

  ¿Los indicadores técnicos calculados en la ventana de gestación contienen información útil para anticipar los movimientos que se expanden y se monetizan más adelante?

Por lo tanto:
- El IC se calcula exclusivamente en la ventana de gestación.
- Los targets ya incorporan, de forma implícita, el desfase temporal hacia la ventana de expansión.
- La coherencia temporal y la ausencia de data leakage quedan garantizadas por construcción.

### **1.4. Conclusión sintética**

La lógica del point 8 define dónde nace la información y dónde se ejecuta la operación.

El cálculo del IC, en el stage_04, cuantifica qué tan informativa es esa ventana de gestación respecto de los resultados que se materializan posteriormente.

Ambos enfoques describen el mismo fenómeno desde niveles distintos (operativo vs estadístico) y están plenamente alineados dentro del diseño del pipeline.

## **2. Separación de dataset IS vs OOS**

En esta etapa se realiza la separación del dataset en dos subconjuntos temporales: **in-sample (IS)** y **out-of-sample (OOS)**.  
Esta división se efectúa de manera estrictamente cronológica, respetando el orden temporal de los datos y evitando cualquier tipo de filtración de información futura.

El conjunto **in-sample (IS)** se utiliza para:
- el cálculo y selección de indicadores técnicos,
- el análisis del Information Coefficient (IC),
- la evaluación de correlaciones y redundancias entre features.

El conjunto **out-of-sample (OOS)** se reserva exclusivamente para:
- validar la estabilidad temporal de las relaciones observadas,
- comprobar que la capacidad predictiva de los indicadores no es producto del sobreajuste,
- verificar que las señales seleccionadas mantienen poder explicativo en datos no vistos.

Esta separación es un paso crítico para garantizar la validez estadística del proceso de feature engineering.  
Un indicador solo se considera apto para su uso en el modelo predictivo si demuestra un comportamiento consistente entre IS y OOS, tanto en magnitud como en signo del IC.

De este modo, la selección final de features se basa en criterios de **robustez temporal**, y no únicamente en el desempeño observado dentro del período de entrenamiento.


**Criterio propuesto (ajustable)**

- IS: desde 2019-12-23 hasta 2022-12-31
- OOS: desde 2023-01-01 hasta 2025-06-13

Este split es solo para selección y validación de features, no es el split final de modelado.

In [23]:
# Trabajamos sobre el dataset existente
df = mnq_intraday_labeled.copy()

# Aseguramos tipo datetime
df["date"] = pd.to_datetime(df["date"])

# Definimos fecha de corte IS / OOS
cut_date = pd.to_datetime("2022-12-31")

# Creamos columna de split para Feature Engineering
df["split_fe"] = np.where(
    df["date"] <= cut_date,
    "IS",   # In-Sample (selección de features)
    "OOS",  # Out-Of-Sample (validación de features)
)

# Sobrescribimos el dataset con la nueva columna
mnq_intraday_labeled = df

## **2. Indicadores Técnicos**

Los indicadores técnicos calculados en cada jornada tienen como objetivo capturar dinámicas intradía relevantes del precio y el volumen, tales como momentum, sobrecompra/sobreventa, presión institucional o posibles reversiones. Cada uno aporta información complementaria sobre el comportamiento del mercado a corto plazo. En particular:

### **2.1. Indicadores técnicos individuales**

#### 1. **RSI (Relative Strength Index)**

Mide la fuerza relativa del precio en los últimos períodos (3, 5, 7, 14), oscilando entre 0 y 100.

  - Valores altos indican posibles condiciones de sobrecompra, mientras que valores bajos sugieren sobreventa.
  
  - Calculado sobre los precios de cierre intradía, el RSI es útil para identificar puntos de reversión potenciales en el corto plazo.

In [24]:
def calcular_rsi(df=mnq_intraday_labeled, target='close' ):
  rsi_columns = ['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['rsi_14'] = ta.momentum.RSIIndicator(grupo[target], window=14).rsi()
        grupo['rsi_7'] = ta.momentum.RSIIndicator(grupo[target], window=7).rsi()
        grupo['rsi_5'] = ta.momentum.RSIIndicator(grupo[target], window=5).rsi()
        grupo['rsi_3'] = ta.momentum.RSIIndicator(grupo[target], window=3).rsi()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, rsi_columns

#### 2. **Momentum**

Mide la aceleración reciente del precio mediante la variación porcentual entre el precio actual y el de hace N minutos.

  - Un valor positivo indica una subida reciente, lo que podría sugerir una continuación alcista.

  - Un valor negativo señala presión bajista reciente, potencialmente anticipando una continuación a la baja.

In [25]:
def calcular_momentum(df=mnq_intraday_labeled, target='close' ):
  momentum_columns = ['momentum_10', 'momentum_5','momentum_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['momentum_10'] = grupo[target].pct_change(10)
        grupo['momentum_5'] = grupo[target].pct_change(5)
        grupo['momentum_3'] = grupo[target].pct_change(3)
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, momentum_columns

#### 3. **Relación de volumen actual vs. su promedio reciente**

Compara el volumen actual con su media móvil en distintas ventanas de tiempo: 15, 20 y 30 minutos.

  - Un valor mayor a 1 indica un volumen superior al promedio de la ventana correspondiente, lo que puede reflejar interés creciente o actividad institucional.

  - Un valor menor a 1 sugiere baja actividad o consolidación del precio.

Esta métrica permite detectar aumentos de volumen ("spikes") sin depender del volumen en crudo, y las diferentes ventanas permiten capturar variaciones en la dinámica de corto plazo con distinta sensibilidad.


In [26]:
def calcular_volumen_ratio(df=mnq_intraday_labeled, target='close'):
  volume_ratio_columns = ['volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['volume_ratio_15'] = grupo['volume'] / grupo['volume'].rolling(15).mean()
        grupo['volume_ratio_20'] = grupo['volume'] / grupo['volume'].rolling(20).mean()
        grupo['volume_ratio_30'] = grupo['volume'] / grupo['volume'].rolling(30).mean()
        grupo['volume_ratio_60'] = grupo['volume'] / grupo['volume'].rolling(60).mean()
        grupo['volume_ratio_90'] = grupo['volume'] / grupo['volume'].rolling(90).mean()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

  return df, volume_ratio_columns

#### 4. **MACD diferencial (señal de cruce)**

Representa la diferencia entre la línea MACD y su línea de señal (una media exponencial de sí misma).

  - Un valor positivo y creciente indica momentum alcista.

  - Un valor negativo sugiere presión bajista.
  
Es ampliamente utilizado para detectar giros de tendencia y cambios en la dinámica del mercado.


In [27]:
def calcular_macd(df=mnq_intraday_labeled, target='close'):
    macd_columns = ['macd']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['macd'] = ta.trend.MACD(grupo[target]).macd_diff()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, macd_columns

#### 5. **Distancia del precio actual a su EMA (15, 20 y 30 minutos)**

Mide el desvío porcentual del precio respecto a su media exponencial en diferentes ventanas, y actúa como indicador de sobreextensión o retorno a la media.

  - Si el precio está muy por encima de la EMA, puede anticipar una reversión bajista o una posible aceleración alcista.

  - Si está por debajo, podría indicar agotamiento o presión vendedora.<br>

Esta métrica se expresa como un porcentaje relativo, lo que facilita la comparación entre distintas ventanas temporales y condiciones de mercado.

Usar varias ventanas (15, 20 y 30 minutos) permite capturar diferentes horizontes de reacción del precio frente a su media móvil.


In [28]:
def calcular_ema(df=mnq_intraday_labeled, target='close'):
    ema_columns = ['ema_15', 'ema_20', 'ema_30',  'ema_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['ema_15'] = grupo[target] / grupo[target].ewm(span=15).mean() - 1
        grupo['ema_20'] = grupo[target] / grupo[target].ewm(span=20).mean() - 1
        grupo['ema_30'] = grupo[target] / grupo[target].ewm(span=30).mean() - 1
        grupo['ema_60'] = grupo[target] / grupo[target].ewm(span=60).mean() - 1
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, ema_columns

#### 6. **%K Estocástico**

Mide la posición relativa del precio actual dentro del rango alto-bajo de los últimos n periodos (generalmente 14).

  - Se utiliza para identificar condiciones extremas de sobrecompra o sobreventa.

  - Un valor cercano a 100 indica que el precio está cerca del máximo reciente (potencial sobrecompra), mientras que un valor cercano a 0 indica proximidad al mínimo reciente (posible sobreventa).

Es útil para detectar momentos en los que el precio puede estar excesivamente extendido y susceptible a una reversión.


In [29]:
def calcular_stochastic(df=mnq_intraday_labeled, target='close'):
    stoch_columns = ['stoch_k_14', 'stoch_k_20', 'stoch_k_30']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        stoch_14 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=14, smooth_window=3
        )
        grupo['stoch_k_14'] = stoch_14.stoch()

        stoch_20 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3
        )
        grupo['stoch_k_20'] = stoch_20.stoch()

        stoch_30 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=30, smooth_window=3
        )
        grupo['stoch_k_30'] = stoch_30.stoch()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, stoch_columns


#### 7.**%B de Bollinger (Bollinger Band Percent)**

Indica la posición del precio actual en relación con las bandas de Bollinger, que están construidas alrededor de una media móvil usando desviaciones estándar.

  - Un valor de %B > 1 sugiere que el precio está por encima de la banda superior, lo que podría implicar exceso de optimismo o momentum fuerte.

  - Un valor < 0 indica que está por debajo de la banda inferior, posible señal de pánico o sobreventa extrema.

Este indicador es eficaz para identificar zonas de congestión, breakout o reversiones basadas en la volatilidad reciente.


In [30]:
from ta.volatility import BollingerBands

def calcular_bollinger(df=mnq_intraday_labeled, target='close'):
    '''bollinger_columns = [
        'bb_percent_15_15', 'bb_percent_20_15', 'bb_percent_30_15',
        'bb_percent_15_20', 'bb_percent_20_20', 'bb_percent_30_20',
        'bb_percent_15_25', 'bb_percent_20_25', 'bb_percent_30_25',
    ]'''

    bollinger_columns = [
        'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15',
        'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20',
        'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 1.5
        grupo['bb_15_15'] = BollingerBands(grupo[target], window=15, window_dev=1.5).bollinger_pband()
        grupo['bb_20_15'] = BollingerBands(grupo[target], window=20, window_dev=1.5).bollinger_pband()
        grupo['bb_30_15'] = BollingerBands(grupo[target], window=30, window_dev=1.5).bollinger_pband()
        grupo['bb_60_15'] = BollingerBands(grupo[target], window=60, window_dev=1.5).bollinger_pband()

        # std: 2
        grupo['bb_15_20'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30_20'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60_20'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()

        # std: 2.5
        grupo['bb_15_25'] = BollingerBands(grupo[target], window=15, window_dev=2.5).bollinger_pband()
        grupo['bb_20_25'] = BollingerBands(grupo[target], window=20, window_dev=2.5).bollinger_pband()
        grupo['bb_30_25'] = BollingerBands(grupo[target], window=30, window_dev=2.5).bollinger_pband()
        grupo['bb_60_25'] = BollingerBands(grupo[target], window=60, window_dev=2.5).bollinger_pband()



        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns


In [31]:
def calcular_bollinger_resume(df=mnq_intraday_labeled, target='close'):

    bollinger_columns = [
        'bb_15', 'bb_20', 'bb_30', 'bb_60',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 2
        grupo['bb_15'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns

#### 8. **ATR normalizado (Average True Range / precio)**

Representa la volatilidad absoluta reciente ajustada al nivel del precio.

  - El ATR mide el rango promedio de oscilación de un activo en los últimos n periodos, capturando tanto movimientos bruscos como gaps.

  - Al normalizarlo dividiéndolo por el precio, se obtiene una medida relativa, comparable entre distintos niveles de mercado.

Este indicador es útil para detectar momentos de alta o baja volatilidad intradía, que pueden influir en la confiabilidad de otras señales técnicas.


In [32]:
def calcular_atr(df=mnq_intraday_labeled, target='close', windows=[5, 10, 14, 20, 30]):
    atr_columns = []

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        for w in windows:
            col_name = f'atr_norm_{w}'
            atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=w)
            grupo[col_name] = atr.average_true_range() / grupo[target]
            atr_columns.append(col_name)
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, list(set(atr_columns))

In [33]:
def calcular_atr_14(df=mnq_intraday_labeled, target='close'):
    atr_columns = ['atr_norm']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=14)
        grupo['atr'] = atr.average_true_range()
        grupo['atr_norm'] = grupo['atr'] / grupo[target]
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, atr_columns

####  9. **ROC (Rate of Change)**

Calcula la tasa de cambio porcentual del precio con respecto a su valor n minutos atrás.

- Es un indicador de momentum que capta aceleraciones o desaceleraciones recientes del precio.

- Valores positivos indican presión alcista; negativos, presión bajista.

A diferencia del momentum tradicional, el ROC expresa el cambio de forma normalizada y en porcentaje, lo que facilita su interpretación comparativa entre distintos activos o marcos temporales.


In [34]:
def calcular_roc(df=mnq_intraday_labeled, target='close'):
    roc_columns = ['roc_5', 'roc_10', 'roc_20','roc_30','roc_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['roc_5'] = ROCIndicator(close=grupo[target], window=5).roc()
        grupo['roc_10'] = ROCIndicator(close=grupo[target], window=10).roc()
        grupo['roc_20'] = ROCIndicator(close=grupo[target], window=20).roc()
        grupo['roc_30'] = ROCIndicator(close=grupo[target], window=30).roc()
        grupo['roc_60'] = ROCIndicator(close=grupo[target], window=60).roc()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, roc_columns

### **2.2. Cálculo de indicadores técnicos**

Calculamos los indicadores técnicos

In [35]:
import os
import glob
import pandas as pd

# ============================================================
# Cargar parquet existente o calcular indicadores y guardar
# ============================================================

PROCESSED_DIR = "/content/drive/MyDrive/neural_profit/data/processed"
PARQUET_NAME = "mnq_intraday_with_indicators.parquet"
PARQUET_PATH = os.path.join(PROCESSED_DIR, PARQUET_NAME)

os.makedirs(PROCESSED_DIR, exist_ok=True)

# 1) Si existe el parquet (o alguno compatible), cargarlo
if os.path.exists(PARQUET_PATH):
    mnq_intraday_with_indicators = pd.read_parquet(PARQUET_PATH)
    print(f"[OK] Cargado: {PARQUET_PATH}")
    indicator_columns = mnq_intraday_with_indicators.columns.tolist()


    # Asegurar que el índice esté en formato datetime
    mnq_intraday_with_indicators.index = pd.to_datetime(mnq_intraday_with_indicators.index)


    columns_to_remove = [
    'date', 'minute_of_day', 'open', 'high', 'low', 'close', 'volume',
    'delta_60', 'delta_90', 'split_fe'
      ]

    indicator_columns = [
        col for col in indicator_columns if col not in columns_to_remove
    ]

else:
    # Fallback: si no existe el nombre exacto, intenta encontrar algún parquet similar
    candidates = sorted(glob.glob(os.path.join(PROCESSED_DIR, "*with_indicators*.parquet")))
    if candidates:
        mnq_intraday_with_indicators = pd.read_parquet(candidates[-1])
        print(f"[OK] Cargado (fallback): {candidates[-1]}")
    else:
        # 2) Calcular indicadores
        mnq_intraday_with_indicators = mnq_intraday_labeled.copy()
        print(f"[OK] Calculando indicadores técnicos")

        mnq_intraday_with_indicators, rsi_columns = calcular_rsi(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, momentum_columns = calcular_momentum(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, volume_ratio_columns = calcular_volumen_ratio(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, macd_columns = calcular_macd(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, ema_columns = calcular_ema(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, stoch_columns = calcular_stochastic(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, bollinger_columns = calcular_bollinger(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, atr_columns = calcular_atr(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, roc_columns = calcular_roc(mnq_intraday_with_indicators)

        indicator_columns = (
          rsi_columns
          + momentum_columns
          + volume_ratio_columns
          + macd_columns
          + ema_columns
          + stoch_columns
          + bollinger_columns
          + atr_columns
          + roc_columns
          )


        # 3) Guardar parquet final
        mnq_intraday_with_indicators.to_parquet(PARQUET_PATH, index=True)
        print(f"[OK] Calculado y guardado: {PARQUET_PATH}")


[OK] Cargado: /content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_with_indicators.parquet


Construimos el listado de indicadores técnicos

Filtramos todos los NaNs del dataset

In [36]:
# Eliminación de filas con NaN
mnq_intraday_with_indicators = mnq_intraday_with_indicators.dropna()

# Verificación posterior
start_time_full_day, final_time_full_day = info_dataset(mnq_intraday_with_indicators)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 392
	Hora diaria de inicio 07:59
	Hora diaria de final 14:30
	Zona horaria: America/New_York


In [37]:
assert not mnq_intraday_with_indicators.isna().any().any(), \
    "El dataset contiene NaN"

##**3. Calculo de Information Coefficient (IC)**

### **3.1. Target `delta_h`**

En el dataset `mnq_intraday_labeled` se trabaja actualmente con **un único tipo de variable objetivo**, correspondiente a **targets continuos** definidos para distintos horizontes temporales. En particular, se utilizan los targets `delta_h` para **h = 60 y h = 90 minutos**.

Dado que ambos targets comparten la misma naturaleza estadística, el cálculo y la interpretación del **Information Coefficient (IC)** se realizan bajo un **criterio metodológico único y consistente**.

La variable `delta_h` representa el **movimiento futuro absoluto del precio**, medido en puntos, desde un instante $ t $ hasta $ t + h $. Se trata de un target **continuo**, directamente vinculado a la magnitud y dirección del desplazamiento futuro del mercado.

Este tipo de target constituye el caso más **directo y conceptualmente puro** para el análisis mediante IC, ya que permite evaluar si un indicador técnico:

- ordena correctamente la **dirección** de los movimientos futuros, y
- discrimina la **magnitud relativa** de dichos movimientos.

Para este análisis se utiliza como métrica el **coeficiente de correlación de Spearman**, dado que:

- no asume relaciones lineales,
- es robusto frente a valores extremos,
- captura asociaciones **monotónicas**, más realistas en series financieras intradía.

La interpretación del IC es directa:

- un **IC positivo** indica que valores más altos del indicador tienden a asociarse con movimientos futuros mayores,
- un **IC negativo** indica una relación inversa.

Por estas razones, `delta_h` (para h = 60 y h = 90) se adopta como **target principal y único** para el análisis de IC en esta etapa del pipeline.


### **3.2. Criterio metodológico — Evaluación de IC por jornada completa y ventanas horarias**

Con el objetivo de alinear la selección de indicadores técnicos con el enfoque del libro, se adopta el siguiente criterio:

Se calculan **múltiples tablas de Information Coefficient (IC)** sobre distintos recortes temporales del día, manteniendo siempre la separación **IS / OOS**:

- **Jornada completa** (mercado intradía completo)
- **Ventana de gestación 08:00–09:00**
- **Ventana  de ejecución 09:00–10:00**

**Justificación**

Este enfoque es metodológicamente sólido porque:

- Permite identificar **factores estructurales**, es decir, indicadores que mantienen IC OOS positivo a lo largo de toda la jornada.
- Permite detectar **factores dependientes del horario**, cuya capacidad predictiva se concentra en ventanas específicas.
- Evita sesgos, siempre que la **selección inicial se base en la jornada completa** y el análisis por ventanas se utilice como refinamiento posterior.

De este modo, el análisis no optimiza prematuramente por horario, sino que primero prioriza **robustez global**.

---

**Uso de los resultados**

A partir de las distintas `ic_tables`, se pueden realizar los siguientes pasos:

1. Identificar la **intersección** de indicadores con IC OOS positivo en todas las ventanas  
   → *core de factores robustos*.

2. Analizar las **diferencias entre ventanas**  
   → detección de features condicionadas por tramo horario.

3. Decidir estrategias posteriores:
   - activación o ponderación de features según el horario, o
   - uso de un único modelo con contexto temporal explícito, o
   - modelos específicos por ventana (etapa posterior).

---

**Síntesis**

> Evaluar IC en la jornada completa y en ventanas horarias permite pasar de una  
> **selección global de factores** a un **refinamiento operativo**,  
> sin romper la lógica metodológica del libro ni introducir sobreajuste.


### **3.3. Implementación de cálculo de IC**

#### **3.3.1. Código**

In [38]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# =========================
# Utilidades base
# =========================

def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    """IC Spearman entre x e y, ignorando NaNs."""
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    return spearmanr(x[mask], y[mask]).correlation


def filter_time_window(df: pd.DataFrame, start: str = "07:59", end: str = "14:30") -> pd.DataFrame:
    """Filtra por ventana horaria intradía. Requiere DatetimeIndex."""
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice del DataFrame debe ser DatetimeIndex para usar between_time().")
    return df.between_time(start, end)


def daily_ic(df: pd.DataFrame, indicator_col: str, target_col: str, date_col: str = "date") -> pd.Series:
    """IC Spearman por día (promediable). Requiere columna date_col."""
    if date_col not in df.columns:
        raise ValueError(f"Falta la columna '{date_col}' en df.")
    return df.groupby(date_col).apply(lambda g: spearman_ic(g[indicator_col], g[target_col]))


# =========================
# Tabla IC (indicadores × horizontes) con IS/OOS
# =========================

def compute_ic_table_is_oos_delta(
    df: pd.DataFrame,
    indicator_columns: list[str],
    horizons: tuple[int, ...] = (60, 90),
    window_start: str = "07:59",
    window_end: str = "14:30",
    use_daily_ic: bool = True,
    split_col: str = "split_fe",     # "IS" / "OOS"
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Calcula IC(indicador, delta_h) para cada indicador y horizonte (60/90),
    separando IS y OOS según split_col.

    Targets esperados:
      - delta_60, delta_90 (formato: f"delta_{h}")

    Devuelve una tabla con:
      - indicator, horizon
      - IC_IS, IC_OOS, delta_oos_minus_is
      - n_pairs_IS, n_pairs_OOS
      - abs_IC_OOS (ranking recomendado) y notas
    """

    # 1) Filtrado horario intradía
    dfw = filter_time_window(df, window_start, window_end).copy()

    # 2) Validaciones de columnas estructurales
    required = {split_col, date_col}
    missing_req = [c for c in required if c not in dfw.columns]
    if missing_req:
        raise ValueError(f"Faltan columnas requeridas: {missing_req}")

    # 3) Subsets IS / OOS
    df_is = dfw[dfw[split_col] == "IS"]
    df_oos = dfw[dfw[split_col] == "OOS"]

    rows = []

    for ind in indicator_columns:
        for h in horizons:
            target_col = f"delta_{h}"

            # Validación de columnas
            missing = [c for c in [ind, target_col] if c not in dfw.columns]
            if missing:
                rows.append({
                    "indicator": ind,
                    "horizon": h,
                    "IC_IS": np.nan,
                    "IC_OOS": np.nan,
                    "delta_oos_minus_is": np.nan,
                    "n_pairs_IS": 0,
                    "n_pairs_OOS": 0,
                    "note": f"missing: {missing}",
                })
                continue

            # --- IS ---
            if use_daily_ic:
                ic_is_series = daily_ic(df_is, ind, target_col, date_col=date_col)
                ic_is = ic_is_series.mean()
            else:
                ic_is = spearman_ic(df_is[ind], df_is[target_col])
            n_pairs_is = int((df_is[ind].notna() & df_is[target_col].notna()).sum())

            # --- OOS ---
            if use_daily_ic:
                ic_oos_series = daily_ic(df_oos, ind, target_col, date_col=date_col)
                ic_oos = ic_oos_series.mean()
            else:
                ic_oos = spearman_ic(df_oos[ind], df_oos[target_col])
            n_pairs_oos = int((df_oos[ind].notna() & df_oos[target_col].notna()).sum())

            rows.append({
                "indicator": ind,
                "horizon": h,
                "IC_IS": ic_is,
                "IC_OOS": ic_oos,
                "delta_oos_minus_is": (ic_oos - ic_is) if (pd.notna(ic_is) and pd.notna(ic_oos)) else np.nan,
                "n_pairs_IS": n_pairs_is,
                "n_pairs_OOS": n_pairs_oos,
                "note": "",
            })

    out = pd.DataFrame(rows)

    # Ranking recomendado: por |IC_OOS| (lo que generaliza)
    out["abs_IC_OOS"] = out["IC_OOS"].abs()
    out = (
        out.sort_values(["horizon", "abs_IC_OOS"], ascending=[True, False])
           .reset_index(drop=True)
    )

    return out


#### **3.3.2. Aplicación**

In [39]:
# ============================================================
# Ventanas temporales para IC (full day + ventanas por hora)
# ============================================================

# Jornada completa (reutiliza los límites generales del dataset intradía)
start_time_full_day
final_time_full_day

# Ventana de gestación (08:00–09:00)
start_time_gestation_window = "08:00"
final_time_gestation_window = "09:00"

# Ventana de ejecución/expansión (09:00–10:00)
start_time_execution_window = "09:00"
final_time_execution_window = "10:00"


In [40]:
import os
import json
import pandas as pd

# ============================================================
# Cache de IC tables (Google Drive)
# ============================================================

CACHE_DIR = "/content/drive/MyDrive/neural_profit/data/processed/ic_tables"
os.makedirs(CACHE_DIR, exist_ok=True)

def _ic_cache_paths(name: str) -> dict:
    """
    Devuelve paths de cache para una ic_table:
      - parquet: datos
      - json: metadatos (parámetros relevantes)
    """
    return {
        "data": os.path.join(CACHE_DIR, f"{name}.parquet"),
        "meta": os.path.join(CACHE_DIR, f"{name}.meta.json"),
    }

def load_or_compute_ic_table(
    *,
    name: str,
    df: pd.DataFrame,
    indicator_columns: list[str],
    horizons: tuple[int, ...],
    window_start: str,
    window_end: str,
    use_daily_ic: bool,
    split_col: str,
    date_col: str,
    force_recompute: bool = False,
) -> pd.DataFrame:
    """
    Carga ic_table desde cache si existe; si no existe, la calcula y la guarda.
    """
    paths = _ic_cache_paths(name)

    # 1) Si existe y no forzamos recálculo → cargar
    if (not force_recompute) and os.path.exists(paths["data"]):
        ic_table = pd.read_parquet(paths["data"])
        print(f"[cache] Loaded: {paths['data']}")
        return ic_table

    # 2) Calcular ic_table
    ic_table = compute_ic_table_is_oos_delta(
        df=df,
        indicator_columns=indicator_columns,
        horizons=horizons,
        window_start=window_start,
        window_end=window_end,
        use_daily_ic=use_daily_ic,
        split_col=split_col,
        date_col=date_col,
    )

    # 3) Guardar datos
    ic_table.to_parquet(paths["data"], index=False)

    # 4) Guardar metadatos mínimos
    meta = {
        "name": name,
        "horizons": list(horizons),
        "window_start": window_start,
        "window_end": window_end,
        "use_daily_ic": use_daily_ic,
        "split_col": split_col,
        "date_col": date_col,
        "n_indicators": len(indicator_columns),
    }
    with open(paths["meta"], "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"[cache] Computed & saved: {paths['data']}")
    return ic_table

In [42]:
# ============================================================
# IC tables IS vs OOS por ventana (con cache en Drive)
# ============================================================

ic_table_full_day = load_or_compute_ic_table(
    name="ic_table_full_day",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_gestation = load_or_compute_ic_table(
    name="ic_table_gestation",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_execution = load_or_compute_ic_table(
    name="ic_table_execution",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)


[cache] Loaded: /content/drive/MyDrive/neural_profit/data/processed/ic_tables/ic_table_full_day.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/processed/ic_tables/ic_table_gestation.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/processed/ic_tables/ic_table_execution.parquet


#### **3.2.3. Resultado**

In [43]:
ic_table_full_day.head(10)

,indicator,horizon,IC_IS,IC_OOS,delta_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,-0.193498,-0.189089,0.004409,281064,229712,,0.189089
1,roc_60,60,-0.187085,-0.177769,0.009315,281064,229712,,0.177769
2,bb_60_15,60,-0.152204,-0.150078,0.002126,281064,229712,,0.150078
3,bb_60_20,60,-0.152204,-0.150078,0.002126,281064,229712,,0.150078
4,bb_60_25,60,-0.152204,-0.150078,0.002126,281064,229712,,0.150078
5,ema_30,60,-0.141006,-0.137613,0.003393,281064,229712,,0.137613
6,roc_30,60,-0.135591,-0.135032,0.000559,281064,229712,,0.135032
7,rsi_14,60,-0.135051,-0.134746,0.000304,281064,229712,,0.134746
8,stoch_k_30,60,-0.122496,-0.127464,-0.004968,281064,229712,,0.127464
9,bb_30_15,60,-0.113529,-0.117261,-0.003733,281064,229712,,0.117261


In [44]:
ic_table_gestation.head(10)

,indicator,horizon,IC_IS,IC_OOS,delta_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,-0.354751,-0.448799,-0.094048,43737,35746,,0.448799
1,ema_30,60,-0.320781,-0.415954,-0.095173,43737,35746,,0.415954
2,roc_30,60,-0.308784,-0.395103,-0.086319,43737,35746,,0.395103
3,bb_60_15,60,-0.307649,-0.389307,-0.081658,43737,35746,,0.389307
4,bb_60_20,60,-0.307649,-0.389307,-0.081658,43737,35746,,0.389307
5,bb_60_25,60,-0.307649,-0.389307,-0.081658,43737,35746,,0.389307
6,rsi_14,60,-0.299523,-0.387252,-0.087729,43737,35746,,0.387252
7,roc_60,60,-0.318990,-0.383652,-0.064662,43737,35746,,0.383652
8,ema_20,60,-0.292266,-0.381854,-0.089588,43737,35746,,0.381854
9,roc_20,60,-0.277609,-0.373835,-0.096226,43737,35746,,0.373835


### **3.3. Separación por horizonte**

In [45]:
# ============================================================
# Separación por horizonte – jornada completa
# ============================================================

ic_60_full_day = ic_table_full_day[ic_table_full_day["horizon"] == 60].copy()
ic_90_full_day = ic_table_full_day[ic_table_full_day["horizon"] == 90].copy()

# ============================================================
# Separación por horizonte – ventana de gestación
# ============================================================

ic_60_gestation = ic_table_gestation[ic_table_gestation["horizon"] == 60].copy()
ic_90_gestation = ic_table_gestation[ic_table_gestation["horizon"] == 90].copy()

# ============================================================
# Separación por horizonte – ventana de ejecución / expansión
# ============================================================

ic_60_execution = ic_table_execution[ic_table_execution["horizon"] == 60].copy()
ic_90_execution = ic_table_execution[ic_table_execution["horizon"] == 90].copy()


### **3.4. Primer filtro: fuerza miníma de señal**

Para el análisis intradía se adopta el siguiente criterio empírico de interpretación del Information Coefficient (IC):

- |IC| < 0.02 → ruido
- 0.02 ≤ |IC| < 0.05 → débil
- 0.05 ≤ |IC| < 0.10 → moderado
- |IC| ≥ 0.10 → fuerte

Dado que el objetivo de este stage es identificar señales con capacidad predictiva real, se descartan aquellas cuyo |IC| se encuentra por debajo del umbral de relevancia. En consecuencia, se conservan únicamente los indicadores que presentan una señal fuerte, definida como:

$$|IC_Δ|≥0.10$$

Este primer filtro elimina indicadores dominados por ruido y reduce el espacio de features a un conjunto con relación estadísticamente significativa respecto a la magnitud del movimiento futuro.

In [46]:
# ============================================================
# Umbral mínimo de relevancia estadística (|IC_OOS|)
# ============================================================

IC_TH = 0.10  # umbral de señal fuerte


# ============================================================
# Jornada completa (full day)
# ============================================================

ic_60_relevant_full_day = (
    ic_60_full_day[ic_60_full_day["abs_IC_OOS"] >= IC_TH].copy()
)

ic_90_relevant_full_day = (
    ic_90_full_day[ic_90_full_day["abs_IC_OOS"] >= IC_TH].copy()
)


# ============================================================
# Ventana de gestación
# ============================================================

ic_60_relevant_gestation = (
    ic_60_gestation[ic_60_gestation["abs_IC_OOS"] >= IC_TH].copy()
)

ic_90_relevant_gestation = (
    ic_90_gestation[ic_90_gestation["abs_IC_OOS"] >= IC_TH].copy()
)


# ============================================================
# Ventana de ejecución / expansión
# ============================================================

ic_60_relevant_execution = (
    ic_60_execution[ic_60_execution["abs_IC_OOS"] >= IC_TH].copy()
)

ic_90_relevant_execution = (
    ic_90_execution[ic_90_execution["abs_IC_OOS"] >= IC_TH].copy()
)


In [47]:
# ============================================================
# Helper: extraer lista ordenada y única de indicadores
# ============================================================

def extract_indicator_list(df: pd.DataFrame) -> list[str]:
    """
    Extrae una lista única, ordenada alfabéticamente, de la columna 'indicator'.
    """
    return (
        df["indicator"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )


# ============================================================
# Jornada completa (full day)
# ============================================================

ti_ic_60_relevant_list_full_day = extract_indicator_list(
    ic_60_relevant_full_day
)

ti_ic_90_relevant_list_full_day = extract_indicator_list(
    ic_90_relevant_full_day
)

# ============================================================
# Ventana de gestación
# ============================================================

ti_ic_60_relevant_list_gestation = extract_indicator_list(
    ic_60_relevant_gestation
)

ti_ic_90_relevant_list_gestation = extract_indicator_list(
    ic_90_relevant_gestation
)


# ============================================================
# Ventana de ejecución / expansión
# ============================================================

ti_ic_60_relevant_list_execution = extract_indicator_list(
    ic_60_relevant_execution
)

ti_ic_90_relevant_list_execution = extract_indicator_list(
    ic_90_relevant_execution
)


### **3.5. Conclusión preliminar**

In [48]:
print("Full day H=60:", ti_ic_60_relevant_list_full_day)
print("Full day H=90:", ti_ic_90_relevant_list_full_day)
print("Gestation H=60:", ti_ic_60_relevant_list_gestation)
print("Gestation H=90:", ti_ic_90_relevant_list_gestation)
print("Execution H=60:", ti_ic_60_relevant_list_execution)
print("Execution H=90:", ti_ic_90_relevant_list_execution)

Full day H=60: ['bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'roc_20', 'roc_30', 'roc_60', 'rsi_14', 'stoch_k_20', 'stoch_k_30']
Full day H=90: ['atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'atr_norm_5', 'bb_15_15', 'bb_15_20', 'bb_15_25', 'bb_20_15', 'bb_20_20', 'bb_20_25', 'bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'momentum_10', 'roc_10', 'roc_20', 'roc_30', 'roc_60', 'rsi_14', 'rsi_5', 'rsi_7', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30']
Gestation H=60: ['bb_15_15', 'bb_15_20', 'bb_15_25', 'bb_20_15', 'bb_20_20', 'bb_20_25', 'bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'macd', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_10', 'roc_20', 'roc_30', 'roc_5', 'roc_60', 'rsi_14', 'rsi_3', 'rsi_5', 'rsi_7', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30']
Gestation H=90: ['bb_15_1

Comentarios sobre los resultados del filtrado por IC

1) La selección es muy amplia  
Aparecen muchas variantes de una misma familia de indicadores (Bandas de Bollinger, EMA, ROC, RSI, Stochastic). Esto es esperable en una primera pasada basada únicamente en IC, pero indica una **alta redundancia** que aún debe ser depurada.

2) Full day muestra un comportamiento más conservador  
Para el horizonte H=60, el conjunto es más reducido y está dominado por:
    - indicadores de tendencia (EMA),
    - indicadores de magnitud del movimiento (ROC),
    - medidas de volatilidad implícita (Bandas de Bollinger),
    - osciladores clásicos (RSI, Stochastic).  

    Esto es coherente con un análisis sobre toda la jornada, donde solo sobreviven señales más estables y persistentes.

3) Gestation y execution capturan señales de corto plazo  
    En ambas ventanas aparecen de forma sistemática:
    - indicadores de momentum (`momentum_3`, `momentum_5`, `momentum_10`),
    - ROC de horizontes cortos,
    - MACD.  

    Esto confirma que en estos tramos horarios el mercado es más **direccional y reactivo**, y que las señales de corto plazo ganan relevancia.

4) El horizonte H=90 incorpora explícitamente riesgo y volatilidad  

    En el análisis full day para H=90 aparecen múltiples variantes de `atr_norm_*`, lo cual es coherente con horizontes más largos, donde la **escala del movimiento** y el régimen de volatilidad resultan más importantes que el timing fino.

5) Similitud entre gestation y execution

    Las listas resultantes para gestation y execution son casi idénticas. Esto sugiere:
    - persistencia del régimen intradía,
    - o bien que aún no se ha aplicado un filtro fuerte de correlación y clustering.  

    No constituye un problema, sino una señal clara de que el siguiente paso metodológico es necesario.

6) Siguientes pasos

    Este resultado no debe utilizarse todavía para el entrenamiento de modelos. Corresponde ahora:
    - agrupar indicadores por familias,
    - eliminar redundancias mediante análisis de correlación,
    - seleccionar un único representante por cluster y por rol.

En síntesis, el IC está cumpliendo su función al detectar señales con capacidad predictiva.

El paso siguiente consiste en **reducir la dimensionalidad con criterio**, priorizando robustez y complementariedad, en lugar de aumentar el número de indicadores.


## **4. Correlación**

### **4.1. Correlación de indicadores técnicos**

Una vez identificado el conjunto de indicadores técnicos relevantes a partir del análisis de Information Coefficient (IC), se procede a estudiar la estructura de dependencia existente entre ellos.

El objetivo de este análisis es cuantificar el grado de colinealidad y redundancia informativa dentro del conjunto seleccionado, así como comprender cómo se relacionan entre sí las distintas familias de indicadores técnicos (tendencia, momentum, magnitud y volatilidad).

Para ello, se calcula la matriz de correlación de Spearman promedio por día, restringida al mismo contexto temporal utilizado en el análisis de IC. En particular, el estudio se realiza de manera diferenciada para la jornada completa y para las ventanas horarias de gestación (08:00–09:00) y de ejecución/expansión (09:00–10:00). Este enfoque permite capturar relaciones monótonas, robustas frente a outliers, y evaluar la estabilidad de dichas relaciones bajo distintos regímenes intradía.

El uso de correlaciones promedio por jornada preserva la consistencia intradía del comportamiento del mercado y evita que episodios aislados dominen la estimación de dependencia entre indicadores.

Este análisis constituye un paso clave previo a la resolución de clusters de redundancia y a la selección final de features, asegurando que el conjunto de variables utilizado en los modelos predictivos sea informativamente complementario y estadísticamente robusto.


#### **4.1.1. Implementación para calculo de Correlación**

In [49]:
import numpy as np
import pandas as pd

def daily_spearman_corr_matrix(
    df: pd.DataFrame,
    indicator_columns: list[str],
    window_start: str,
    window_end: str,
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Matriz Spearman PROMEDIO por día, restringida a la ventana horaria.

    Requisitos:
      - df.index: DatetimeIndex
      - df contiene columna date_col
      - indicator_columns existen en df
    """
    # 0) Validaciones rápidas
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("df.index debe ser DatetimeIndex para usar between_time().")

    if not indicator_columns:
        raise ValueError("indicator_columns está vacío. No hay indicadores para correlacionar.")

    if date_col not in df.columns:
        raise ValueError(f"Falta la columna '{date_col}' en df.")

    # 1) Filtrar ventana horaria
    df_win = df.between_time(window_start, window_end).copy()

    # 2) Validar columnas
    missing = [c for c in indicator_columns if c not in df_win.columns]
    if missing:
        raise ValueError(f"Faltan columnas en df: {missing}")

    # 3) Correlación Spearman por día (matriz)
    daily_corr = []
    for _, g in df_win.groupby(date_col):
        X = g[indicator_columns]
        corr = X.corr(method="spearman")

        # Guardar solo matrices con algo útil
        if corr.notna().values.any():
            daily_corr.append(corr)

    if not daily_corr:
        raise ValueError("No se pudieron calcular correlaciones (ventana vacía, NaNs o sin datos por día).")

    # 4) Promedio (alineado por índices/columnas)
    corr_mean = sum(daily_corr) / len(daily_corr)
    return corr_mean


def top_abs_correlations(
    corr: pd.DataFrame,
    top_n: int = 15,
    min_abs_rho: float = 0.0,
) -> pd.DataFrame:
    """
    Ranking de pares con mayor |rho| (sin duplicados i-j y sin diagonal).
    """
    c = corr.copy()

    # Enmascarar diagonal y triángulo inferior (evita duplicados)
    mask = np.tril(np.ones(c.shape, dtype=bool))
    c = c.mask(mask)

    pairs = (
        c.stack()
         .rename("rho")
         .reset_index()
         .rename(columns={"level_0": "indicator_1", "level_1": "indicator_2"})
    )

    pairs["abs_rho"] = pairs["rho"].abs()
    pairs = pairs[pairs["abs_rho"] >= min_abs_rho]

    return pairs.sort_values("abs_rho", ascending=False).head(top_n).reset_index(drop=True)



#### **4.1.2. Aplicación de cálculo de Correlación**

In [50]:
# ============================================================
# Configuración de ventanas (alineado con full_day / gestation / execution)
# ============================================================

WINDOWS = {
    "full_day":   (start_time_full_day, final_time_full_day),
    "gestation":  (start_time_gestation_window, final_time_gestation_window),
    "execution":  (start_time_execution_window, final_time_execution_window),
}

# ============================================================
# Listas de indicadores relevantes por horizonte y ventana
# ============================================================

INDICATORS = {
    ("full_day", 60):   ti_ic_60_relevant_list_full_day,
    ("full_day", 90):   ti_ic_90_relevant_list_full_day,
    ("gestation", 60):  ti_ic_60_relevant_list_gestation,
    ("gestation", 90):  ti_ic_90_relevant_list_gestation,
    ("execution", 60):  ti_ic_60_relevant_list_execution,
    ("execution", 90):  ti_ic_90_relevant_list_execution,
}

# ============================================================
# Calcular matrices de correlación Spearman promedio por día
# ============================================================

corr_mean = {}

for (window_name, horizon), indicators in INDICATORS.items():
    w_start, w_end = WINDOWS[window_name]

    # Si por algún motivo la lista está vacía, evitamos errores
    if not indicators:
        print(f"[warn] Lista vacía: window={window_name}, horizon={horizon}. Se omite.")
        continue

    corr_mean[(window_name, horizon)] = daily_spearman_corr_matrix(
        df=mnq_intraday_with_indicators,
        indicator_columns=indicators,
        window_start=w_start,
        window_end=w_end,
        date_col="date",
    )

# ============================================================
# Acceso a resultados (variables explícitas, naming consistente)
# ============================================================

corr_mean_ic_60_relevant_full_day   = corr_mean[("full_day", 60)]
corr_mean_ic_90_relevant_full_day   = corr_mean[("full_day", 90)]

corr_mean_ic_60_relevant_gestation  = corr_mean[("gestation", 60)]
corr_mean_ic_90_relevant_gestation  = corr_mean[("gestation", 90)]

corr_mean_ic_60_relevant_execution  = corr_mean[("execution", 60)]
corr_mean_ic_90_relevant_execution  = corr_mean[("execution", 90)]


In [61]:
corr_mean_ic_60_relevant_gestation.head(10)

,bb_15_15,bb_15_20,bb_15_25,bb_20_15,bb_20_20,bb_20_25,bb_30_15,bb_30_20,bb_30_25,bb_60_15,...,roc_30,roc_5,roc_60,rsi_14,rsi_3,rsi_5,rsi_7,stoch_k_14,stoch_k_20,stoch_k_30
bb_15_15,1.000000,1.000000,1.000000,0.957430,0.957430,0.957430,0.866192,0.866192,0.866192,0.720876,...,0.456075,0.809139,0.357607,0.857105,0.864398,0.941403,0.946032,0.931028,0.879016,0.806022
bb_15_20,1.000000,1.000000,1.000000,0.957430,0.957430,0.957430,0.866192,0.866192,0.866192,0.720876,...,0.456075,0.809139,0.357607,0.857105,0.864398,0.941403,0.946032,0.931028,0.879016,0.806022
bb_15_25,1.000000,1.000000,1.000000,0.957430,0.957430,0.957430,0.866192,0.866192,0.866192,0.720876,...,0.456075,0.809139,0.357607,0.857105,0.864398,0.941403,0.946032,0.931028,0.879016,0.806022
bb_20_15,0.957430,0.957430,0.957430,1.000000,1.000000,1.000000,0.935907,0.935907,0.935907,0.791534,...,0.526393,0.758718,0.400203,0.905636,0.801930,0.909034,0.943390,0.908040,0.922509,0.860665
bb_20_20,0.957430,0.957430,0.957430,1.000000,1.000000,1.000000,0.935907,0.935907,0.935907,0.791534,...,0.526393,0.758718,0.400203,0.905636,0.801930,0.909034,0.943390,0.908040,0.922509,0.860665
bb_20_25,0.957430,0.957430,0.957430,1.000000,1.000000,1.000000,0.935907,0.935907,0.935907,0.791534,...,0.526393,0.758718,0.400203,0.905636,0.801930,0.909034,0.943390,0.908040,0.922509,0.860665
bb_30_15,0.866192,0.866192,0.866192,0.935907,0.935907,0.935907,1.000000,1.000000,1.000000,0.883662,...,0.647719,0.683207,0.463443,0.941387,0.714264,0.841549,0.904057,0.836524,0.895706,0.913111
bb_30_20,0.866192,0.866192,0.866192,0.935907,0.935907,0.935907,1.000000,1.000000,1.000000,0.883662,...,0.647719,0.683207,0.463443,0.941387,0.714264,0.841549,0.904057,0.836524,0.895706,0.913111
bb_30_25,0.866192,0.866192,0.866192,0.935907,0.935907,0.935907,1.000000,1.000000,1.000000,0.883662,...,0.647719,0.683207,0.463443,0.941387,0.714264,0.841549,0.904057,0.836524,0.895706,0.913111
bb_60_15,0.720876,0.720876,0.720876,0.791534,0.791534,0.791534,0.883662,0.883662,0.883662,1.000000,...,0.775579,0.571480,0.609078,0.915195,0.588881,0.717071,0.796051,0.705174,0.779795,0.847808


In [60]:
corr_mean_ic_90_relevant_gestation.head(10)

,bb_15_15,bb_15_20,bb_15_25,bb_20_15,bb_20_20,bb_20_25,bb_30_15,bb_30_20,bb_30_25,bb_60_15,...,roc_30,roc_5,roc_60,rsi_14,rsi_3,rsi_5,rsi_7,stoch_k_14,stoch_k_20,stoch_k_30
bb_15_15,1.000000,1.000000,1.000000,0.957430,0.957430,0.957430,0.866192,0.866192,0.866192,0.720876,...,0.456075,0.809139,0.357607,0.857105,0.864398,0.941403,0.946032,0.931028,0.879016,0.806022
bb_15_20,1.000000,1.000000,1.000000,0.957430,0.957430,0.957430,0.866192,0.866192,0.866192,0.720876,...,0.456075,0.809139,0.357607,0.857105,0.864398,0.941403,0.946032,0.931028,0.879016,0.806022
bb_15_25,1.000000,1.000000,1.000000,0.957430,0.957430,0.957430,0.866192,0.866192,0.866192,0.720876,...,0.456075,0.809139,0.357607,0.857105,0.864398,0.941403,0.946032,0.931028,0.879016,0.806022
bb_20_15,0.957430,0.957430,0.957430,1.000000,1.000000,1.000000,0.935907,0.935907,0.935907,0.791534,...,0.526393,0.758718,0.400203,0.905636,0.801930,0.909034,0.943390,0.908040,0.922509,0.860665
bb_20_20,0.957430,0.957430,0.957430,1.000000,1.000000,1.000000,0.935907,0.935907,0.935907,0.791534,...,0.526393,0.758718,0.400203,0.905636,0.801930,0.909034,0.943390,0.908040,0.922509,0.860665
bb_20_25,0.957430,0.957430,0.957430,1.000000,1.000000,1.000000,0.935907,0.935907,0.935907,0.791534,...,0.526393,0.758718,0.400203,0.905636,0.801930,0.909034,0.943390,0.908040,0.922509,0.860665
bb_30_15,0.866192,0.866192,0.866192,0.935907,0.935907,0.935907,1.000000,1.000000,1.000000,0.883662,...,0.647719,0.683207,0.463443,0.941387,0.714264,0.841549,0.904057,0.836524,0.895706,0.913111
bb_30_20,0.866192,0.866192,0.866192,0.935907,0.935907,0.935907,1.000000,1.000000,1.000000,0.883662,...,0.647719,0.683207,0.463443,0.941387,0.714264,0.841549,0.904057,0.836524,0.895706,0.913111
bb_30_25,0.866192,0.866192,0.866192,0.935907,0.935907,0.935907,1.000000,1.000000,1.000000,0.883662,...,0.647719,0.683207,0.463443,0.941387,0.714264,0.841549,0.904057,0.836524,0.895706,0.913111
bb_60_15,0.720876,0.720876,0.720876,0.791534,0.791534,0.791534,0.883662,0.883662,0.883662,1.000000,...,0.775579,0.571480,0.609078,0.915195,0.588881,0.717071,0.796051,0.705174,0.779795,0.847808


#### **4.1.3. Observaciones clave (correlación Spearman promedio)**

**Full day:**

- Hay mucha redundancia: Bollinger, EMA, RSI, Stochastic y momentum corto se parecen entre sí.
- roc_60 aporta información más independiente.
- En H=90, atr_norm_* captura volatilidad, un eje distinto.
- Conclusión: quedarse con un indicador por familia y combinar solo tendencia + magnitud + volatilidad.

**Gestation:**

- Bloque Bollinger (bb_*): muy redundante (muchos 1.00 y >0.93). Con uno alcanza.
- Bloque EMA (ema_15/20/30/60): alta colinealidad; ema_60 está muy ligado al resto (≈0.70–0.92). No conviene usar varias EMA juntas.
- Bloque RSI/Stoch: también muy correlacionados (RSI entre sí ≈0.95–0.98; Stoch entre sí ≈0.92+). Son casi la misma información.
- Momentum: momentum_5 es muy cercano a roc_5 (≈1.00) y momentum_10≈roc_10 (≈1.00). Hay solapamiento fuerte ahí.
- ROC: los ROC cortos (20/30) están correlacionados entre sí (≈0.69) y con roc_60 en forma moderada (≈0.59). roc_60 es el más “distinto”.
- MACD: correlación moderada con el resto; no se ve como “aporte único” claro.

  Resumen práctico: en gestación, el set está muy redundante; si quiere simplificar, deje 1 representante por familia (una EMA, un ROC “largo”, un momentum/ROC corto, y evitar duplicados RSI/Stoch/Bollinger).


**Execution:**

- La redundancia es aún mayor que en la jornada completa: Bollinger, RSI, Stochastic y EMA están fuertemente correlacionados.
- roc_60 vuelve a ser el menos redundante dentro de su familia.
- Los momentum_* aportan algo distinto, pero entre ellos también hay solapamiento.
- MACD no agrega información claramente independiente.
- Conclusión: en ejecución conviene simplificar fuerte: un representante de tendencia, uno de momentum y uno de magnitud; el resto es ruido redundante.



###**4.2. Clusters de Correlación**


Dado el alto nivel de colinealidad observado entre numerosos indicadores (y la redundancia entre distintas parametrizaciones), se construyen clusters de indicadores para agrupar señales equivalentes y obtener un set final más eficiente. Este procedimiento permite:

- agrupar indicadores que capturan esencialmente la misma información,
- reducir dimensionalidad sin perder señal,
- seleccionar representantes por cluster (o combinar señales) para disminuir el riesgo de sobreajuste,
- mejorar la interpretabilidad y robustez del conjunto de features.

Definición conceptual (en términos de grafo):

- Cada indicador es un nodo.

- Existe una arista entre dos indicadores i,j si su correlación absoluta cumple:   $|𝜌_{ij} \ge 0.85|$

- Cada componente conexa del grafo es un cluster informativo.

En otras palabras, cada cluster representa un grupo de indicadores que comparten prácticamente la misma información y, por lo tanto, pueden tratarse como una misma “familia” operativa a efectos de selección de features.

#### **4.2.1. Código para construcción de clusters**

Este código:
- toma la matriz de correlación media (corr_mean_ic_h_relevant)
- detecta clusters automáticamente
- devuelve un DataFrame limpio para inspección

In [62]:
import numpy as np
import pandas as pd
import networkx as nx


def build_correlation_clusters(
    corr: pd.DataFrame,
    threshold: float = 0.85,
    *,
    use_upper_triangle: bool = True,
) -> pd.DataFrame:
    """
    Construye clusters de indicadores basados en alta correlación (|rho| >= threshold).

    Idea:
      - Cada indicador = nodo
      - Conectamos dos nodos con una arista si |rho| >= threshold
      - Cada componente conexa del grafo = cluster de redundancia

    Parameters
    ----------
    corr : pd.DataFrame
        Matriz de correlación (Spearman promedio), index = columns = indicadores.
    threshold : float
        Umbral de |rho| para considerar redundancia.
    use_upper_triangle : bool
        Si True, recorre solo el triángulo superior (más eficiente y sin duplicar aristas).

    Returns
    -------
    clusters_df : pd.DataFrame
        Columnas:
          - cluster_id
          - indicator
          - n_in_cluster
    """

    # -------------------------
    # 0) Validaciones mínimas
    # -------------------------
    if not isinstance(corr, pd.DataFrame) or corr.empty:
        raise ValueError("corr debe ser un DataFrame no vacío.")

    if corr.shape[0] != corr.shape[1]:
        raise ValueError("corr debe ser una matriz cuadrada (NxN).")

    if list(corr.index) != list(corr.columns):
        # No es obligatorio, pero evita sorpresas
        corr = corr.copy()
        corr = corr.loc[corr.index, corr.index]

    indicators = corr.columns.tolist()

    # -------------------------
    # 1) Crear grafo
    # -------------------------
    G = nx.Graph()
    G.add_nodes_from(indicators)

    # -------------------------
    # 2) Agregar aristas (|rho| >= threshold)
    # -------------------------
    if use_upper_triangle:
        # Recorre solo i < j (sin duplicar)
        for a, i in enumerate(indicators):
            for j in indicators[a + 1:]:
                rho = corr.at[i, j]
                if pd.notna(rho) and abs(rho) >= threshold:
                    G.add_edge(i, j, weight=float(rho))
    else:
        # Recorre todo (más lento, pero explícito)
        for i in indicators:
            for j in indicators:
                if i == j:
                    continue
                rho = corr.at[i, j]
                if pd.notna(rho) and abs(rho) >= threshold:
                    G.add_edge(i, j, weight=float(rho))

    # -------------------------
    # 3) Componentes conexas = clusters
    # -------------------------
    components = list(nx.connected_components(G))

    # -------------------------
    # 4) Armar DataFrame de salida
    # -------------------------
    records = []
    for cluster_id, comp in enumerate(components):
        comp = sorted(comp)
        n = len(comp)
        for ind in comp:
            records.append({"cluster_id": cluster_id, "indicator": ind, "n_in_cluster": n})

    clusters_df = (
        pd.DataFrame(records)
          .sort_values(["n_in_cluster", "cluster_id", "indicator"], ascending=[False, True, True])
          .reset_index(drop=True)
    )

    return clusters_df


#### **4.2.2. Aplicación**

- `cluster_id`: identifica cada grupo informativo
- `n_in_cluster`:
  - 1 → indicador independiente
  - $>$ 1 → redundancia fuerte

In [63]:
# ============================================================
# Cálculo de clusters (sin repetir llamadas)
# ============================================================

THRESH = 0.85

# Diccionario de matrices de correlación ya calculadas (de la etapa anterior)
CORR_MATS = {
    ("full_day", 60): corr_mean_ic_60_relevant_full_day,
    ("full_day", 90): corr_mean_ic_90_relevant_full_day,
    ("gestation", 60): corr_mean_ic_60_relevant_gestation,
    ("gestation", 90): corr_mean_ic_90_relevant_gestation,
    ("execution", 60): corr_mean_ic_60_relevant_execution,
    ("execution", 90): corr_mean_ic_90_relevant_execution,
}

# Calcula todos los clusters en un loop
clusters = {}
for (window_name, horizon), corr_mat in CORR_MATS.items():
    clusters[(window_name, horizon)] = build_correlation_clusters(
        corr=corr_mat,
        threshold=THRESH,
        use_upper_triangle=True,  # más eficiente
    )

# Acceso con nombres “tipo variable” (como usted venía usando)
clusters_ic_60_full_day  = clusters[("full_day", 60)]
clusters_ic_90_full_day  = clusters[("full_day", 90)]
clusters_ic_60_gestation = clusters[("gestation", 60)]
clusters_ic_90_gestation = clusters[("gestation", 90)]
clusters_ic_60_execution = clusters[("execution", 60)]
clusters_ic_90_execution = clusters[("execution", 90)]

In [75]:
#print('clusters_ic_60_execution:')
#clusters_ic_60_execution

In [76]:
#print('clusters_ic_90_execution:')
#clusters_ic_90_execution

#### **4.2.3. Observaciones de clusters**

**Full Day**

  `clusters_ic_60_full_day`
  
  - Se observa un único cluster dominante (15 indicadores) que agrupa Bollinger Bands, EMA, ROC de corto plazo, RSI y Stochastic, lo que indica un alto nivel de redundancia informativa.  
  - El indicador `roc_60` queda aislado en su propio cluster, señalando que aporta información distinta al resto.  
  - Interpretación: para el horizonte H=60, la mayoría de los indicadores capturan la misma estructura del mercado; resulta suficiente seleccionar un indicador representativo del cluster principal y complementar con `roc_60`.

  `clusters_ic_90_full_day`

  - Se identifica un cluster muy grande (26 indicadores) que reúne Bollinger Bands, EMA, momentum, ROC cortos, RSI y Stochastic, evidenciando colinealidad extrema.  
  - Los indicadores de volatilidad (`atr_norm_*`) forman un cluster separado (5 indicadores), aportando información diferente al precio y al momentum.  
  - El indicador `roc_60` vuelve a aparecer aislado, confirmando su rol como señal estructural independiente y robusta.  
  - Interpretación: para el horizonte H=90, es conveniente utilizar un indicador del cluster principal, uno representativo de volatilidad (ATR) y `roc_60`.

  Resumen operativo

  - El análisis de clusters confirma que el conjunto completo de indicadores presenta alta redundancia.  
  - `roc_60` emerge como un indicador clave y no redundante en ambos horizontes.  
  - Para H=90, la volatilidad medida mediante ATR introduce una dimensión adicional que no resulta relevante en H=60.


**Gestation**

`clusters_ic_60_gestation`

- Existe un cluster dominante muy grande (23 indicadores) que agrupa Bollinger Bands, EMA, RSI y Stochastic, indicando una fuerte redundancia estructural durante la ventana de gestación.  
- Un segundo cluster pequeño (3 indicadores) reúne `macd`, `momentum_10` y `roc_10`, sugiriendo una familia de señales de momentum/velocidad distinta al bloque principal.  
- Aparecen clusters adicionales de tamaño reducido:
  - `momentum_5` y `roc_5` forman un cluster propio (alta similitud entre ambos).
  - `momentum_3`, `roc_20`, `roc_30` y `roc_60` quedan aislados, aportando información diferenciada por escala temporal.  
- Interpretación: en gestación H=60 conviene elegir un único representante del cluster principal y complementar con señales de momentum y ROC de distintos horizontes.

`clusters_ic_90_gestation`

- La estructura de clusters es prácticamente idéntica a H=60, lo que indica estabilidad del régimen informativo en la ventana de gestación.  
- Se mantiene el gran cluster redundante de 23 indicadores de tendencia/oscillators.  
- `macd`, `momentum_10` y `roc_10` vuelven a agruparse, confirmando su rol conjunto como señales de impulso temprano.  
- `roc_60` permanece aislado, reforzando su carácter estructural y no redundante también para H=90.  
- Interpretación: la selección de features para H=90 puede seguir el mismo criterio que H=60, sin necesidad de redefinir familias.

Resumen operativo

- La ventana de gestación presenta alta colinealidad en indicadores clásicos de tendencia y osciladores.  
- Los indicadores de momentum y ROC, especialmente en distintos horizontes, aportan diversidad informativa real.  
- `roc_60` se consolida como un indicador clave, independiente y consistente en ambos horizontes.


**Execution**

`clusters_ic_60_execution`

- Se observa un cluster dominante muy amplio (27 indicadores) que agrupa Bollinger Bands, EMA, MACD, RSI, Stochastic y ROC/ momentum de corto plazo, evidenciando una redundancia extremadamente alta durante la ventana de ejecución.  
- Este comportamiento sugiere que, en esta ventana, muchas señales técnicas están reaccionando simultáneamente al mismo movimiento de precio ya en desarrollo.  
- Aparece un cluster secundario pequeño formado por `momentum_5` y `roc_5`, indicando similitud fuerte entre ambos como señales de muy corto plazo.  
- `momentum_3`, `roc_20` y `roc_60` quedan aislados en clusters individuales, aportando información diferenciada por escala temporal.  
- Interpretación: en ejecución H=60, la mayor parte de los indicadores clásicos no aporta diversidad informativa; conviene seleccionar muy pocos representantes y priorizar horizontes distintos.

`clusters_ic_90_execution`

- La estructura de clusters es idéntica a H=60, lo que confirma que el régimen informativo de la ventana de ejecución es estable e independiente del horizonte del target.  
- Se mantiene el gran cluster redundante de 27 indicadores, reforzando la idea de fuerte colinealidad en fase de ejecución.  
- `momentum_5` y `roc_5` vuelven a agruparse, consolidándose como señales equivalentes.  
- `momentum_3`, `roc_20` y `roc_60` permanecen aislados, confirmando su rol complementario y no redundante.  

Resumen operativo

- La ventana de ejecución presenta la mayor redundancia de todo el análisis.  
- La mayoría de indicadores clásicos describen el mismo estado del mercado una vez iniciado el movimiento.  
- Para modelos predictivos, esta ventana debería utilizarse con un set mínimo de features, privilegiando indicadores estructurales (`ema_60`, `roc_60`) y uno o dos de corto plazo como complemento.


###**4.3. Procesamiento de clusters**


#### **4.3.1. Resolución de cluster dominante**

Una vez identificados los clusters por alta correlación (redundancia), el objetivo es conservar un solo indicador por cluster, eligiendo el que tenga mayor potencia predictiva.

Para eso, dentro de cada `cluster_id` se selecciona el indicador con mayor `abs_IC_delta` (y, en caso de empate, mayor `abs(IC_trade_only)`).

#### **4.3.2. Código**

In [77]:
import numpy as np
import pandas as pd

# ============================================================
# Resolver clusters por IC (adaptado a IS/OOS + ventanas)
# ============================================================
# Cambio clave respecto a su versión anterior:
# - Ya NO usamos IC_delta / abs_IC_delta (viejo esquema).
# - Ahora usamos IC_OOS (y su absoluto) como criterio principal,
#   porque queremos seleccionar indicadores que GENERALICEN fuera de muestra.
#
# Entradas esperadas (ic_relevant):
#   ['indicator','horizon','IC_IS','IC_OOS','abs_IC_OOS','n_pairs_OOS','note', ...]
# Entradas esperadas (clusters_df):
#   ['cluster_id','indicator','n_in_cluster']
#
# Nota:
# - El DataFrame de clusters NO tiene horizon (la correlación fue por lista),
#   así que el "horizon" lo aporta ic_relevant al hacer merge.
# - Para ventana "complete / gestation / expansion", llamaremos esta función
#   con el ic_relevant y clusters_df correspondientes a esa ventana.
# ============================================================

def resolve_clusters_by_ic_oos(
    ic_relevant: pd.DataFrame,
    clusters_df: pd.DataFrame,
    *,
    prefer_ic_is_tiebreak: bool = True,
) -> pd.DataFrame:
    """
    Selecciona 1 indicador por cluster, usando dominancia de IC out-of-sample (OOS).

    Selección:
      1) Mayor abs_IC_OOS
      2) (opcional) Si hay empate: mayor abs(IC_IS) (para preferir señal consistente)
      3) Si persiste: orden alfabético (determinístico)

    Devuelve tabla con:
      - window_name (si se agrega fuera), horizon, cluster_id, n_in_cluster,
        indicator, keep, reason, IC_IS, IC_OOS, abs_IC_OOS, n_pairs_OOS, note
    """

    needed_ic = {"indicator", "horizon", "IC_IS", "IC_OOS", "abs_IC_OOS", "n_pairs_OOS", "note"}
    needed_cl = {"cluster_id", "indicator", "n_in_cluster"}

    if not needed_ic.issubset(ic_relevant.columns):
        raise ValueError(f"ic_relevant debe contener: {sorted(needed_ic)}")
    if not needed_cl.issubset(clusters_df.columns):
        raise ValueError(f"clusters_df debe contener: {sorted(needed_cl)}")

    # Merge: cluster info + IC (incluye horizon)
    df = clusters_df.merge(
        ic_relevant[["indicator", "horizon", "IC_IS", "IC_OOS", "abs_IC_OOS", "n_pairs_OOS", "note"]],
        on="indicator",
        how="left"
    )

    # Validar que todos los indicadores tengan IC_OOS
    missing = df[df["abs_IC_OOS"].isna()]["indicator"].unique().tolist()
    if missing:
        raise ValueError(f"Indicadores en clusters sin IC_OOS en ic_relevant: {missing}")

    out_rows = []

    # Resolver por (horizon, cluster_id) para no mezclar criterios entre H=60 y H=90
    for (h, cid), g in df.groupby(["horizon", "cluster_id"], sort=True):
        g = g.copy()

        # 1) criterio principal: max abs_IC_OOS
        max_abs = g["abs_IC_OOS"].max()
        candidates = g[g["abs_IC_OOS"] == max_abs].copy()

        # 2) desempate: abs(IC_IS) (si se solicita)
        tiebreak_used = False
        if prefer_ic_is_tiebreak and len(candidates) > 1:
            candidates["abs_IC_IS_tb"] = candidates["IC_IS"].abs()
            candidates = candidates.sort_values(["abs_IC_IS_tb", "indicator"], ascending=[False, True])
            best = candidates.iloc[0]
            tiebreak_used = True
        else:
            # 3) determinístico: alfabético si hay varios
            candidates = candidates.sort_values("indicator", ascending=True)
            best = candidates.iloc[0]

        best_indicator = best["indicator"]

        for _, row in g.iterrows():
            keep = (row["indicator"] == best_indicator)

            if keep:
                reason = "max abs_IC_OOS"
                if len(candidates) > 1:
                    if prefer_ic_is_tiebreak and tiebreak_used:
                        reason += " (tiebreak: abs(IC_IS))"
                    else:
                        reason += " (tiebreak: indicator order)"
            else:
                reason = f"redundant (same cluster as {best_indicator})"

            out_rows.append({
                "horizon": int(row["horizon"]),
                "cluster_id": int(cid),
                "n_in_cluster": int(row["n_in_cluster"]),
                "indicator": row["indicator"],
                "keep": bool(keep),
                "reason": reason,
                "IC_IS": float(row["IC_IS"]),
                "IC_OOS": float(row["IC_OOS"]),
                "abs_IC_OOS": float(row["abs_IC_OOS"]),
                "n_pairs_OOS": int(row["n_pairs_OOS"]),
                "note": str(row["note"]) if row["note"] is not None else "",
            })

    resolved = (
        pd.DataFrame(out_rows)
        .sort_values(["horizon", "cluster_id", "keep", "abs_IC_OOS"], ascending=[True, True, False, False])
        .reset_index(drop=True)
    )

    return resolved

In [78]:
# ============================================================
# Orquestación por ventana y horizonte (complete / gestation / expansion)
# ============================================================
# Requiere que ya tenga:
# - ic tables por ventana (output de compute_ic_table_is_oos_delta):
#     ic_table_complete, ic_table_gestation, ic_table_expansion
# - y clusters por ventana/horizonte:
#     clusters[("complete",60)], etc. (como lo armamos antes)
#
# Importante:
# - "ic_table_*" contiene filas para h=60 y h=90.
# - Filtramos por horizon y por umbral |IC_OOS| (si aplica) ANTES de resolver clusters.
# ============================================================

TH_IC = 0.02  # ejemplo razonable para intradía; ajústelo a su criterio

def prepare_ic_relevant(ic_table: pd.DataFrame, horizon: int, th: float) -> pd.DataFrame:
    """
    Filtra la tabla de IC por horizonte y umbral de |IC_OOS|.
    Devuelve un ic_relevant compatible con resolve_clusters_by_ic_oos.
    """
    dfh = ic_table[ic_table["horizon"] == horizon].copy()

    # Asegurar abs_IC_OOS (por si no existiera)
    if "abs_IC_OOS" not in dfh.columns and "IC_OOS" in dfh.columns:
        dfh["abs_IC_OOS"] = dfh["IC_OOS"].abs()

    # Filtrar por umbral (señal mínima OOS)
    dfh = dfh[dfh["abs_IC_OOS"] >= th].copy()

    # Nota: rename por compatibilidad si su tabla usa n_pairs_OOS/n_pairs_IS
    # (si ya los tiene, no hace nada)
    if "n_pairs_OOS" not in dfh.columns and "n_pairs_OOS" in dfh.columns:
        pass

    return dfh


In [79]:
def resolve_window(
    window_name: str,
    ic_table_window: pd.DataFrame,
    clusters_dict: dict[tuple[str, int], pd.DataFrame],
    threshold_ic: float = TH_IC,
) -> dict[int, dict[str, object]]:
    """
    Resuelve clusters para una ventana (complete/gestation/expansion) y devuelve:
      - final_indicators por horizonte
      - resolved tables por horizonte
    """
    result = {}

    for h in (60, 90):
        # 1) ic_relevant para este horizonte
        ic_rel = prepare_ic_relevant(ic_table_window, horizon=h, th=threshold_ic)

        # Si queda vacío, no hay nada que resolver
        if ic_rel.empty:
            result[h] = {
                "final_indicators": [],
                "resolved": pd.DataFrame(),
                "note": f"Sin indicadores que superen el umbral |IC_OOS| >= {threshold_ic} en {window_name} (H={h})"
            }
            continue

        # 2) clusters de esta ventana/horizonte (construidos a partir de los relevantes)
        cl = clusters_dict[(window_name, h)].copy()

        # 3) filtrar clusters a indicadores presentes en ic_rel (safety)
        cl_rel = cl[cl["indicator"].isin(ic_rel["indicator"])].copy()

        # 4) resolver
        resolved = resolve_clusters_by_ic_oos(
            ic_relevant=ic_rel,
            clusters_df=cl_rel,
            prefer_ic_is_tiebreak=True,
        )

        # 5) lista final
        final_indicators = resolved.loc[resolved["keep"], "indicator"].tolist()

        result[h] = {
            "final_indicators": final_indicators,
            "resolved": resolved,
            "note": ""
        }

    return result

#### **4.3.3. Aplicación**

In [82]:
# Suponga que:
# - ic_table_complete, ic_table_gestation, ic_table_expansion ya están calculadas
# - clusters dict ya existe: clusters[(window,h)]

full_day_res = resolve_window("full_day",  ic_table_full_day,  clusters, threshold_ic=TH_IC)
gest_res     = resolve_window("gestation", ic_table_gestation, clusters, threshold_ic=TH_IC)
exe_res      = resolve_window("execution", ic_table_execution, clusters, threshold_ic=TH_IC)

In [83]:
full_day_final_60 = full_day_res[60]["final_indicators"]
full_day_final_90 = full_day_res[90]["final_indicators"]
gestation_final_60 = gest_res[60]["final_indicators"]
gestation_final_90 = gest_res[90]["final_indicators"]
execution_final_60 = exe_res[60]["final_indicators"]
execution_final_90 = exe_res[90]["final_indicators"]

In [84]:
# Para ver qué se conserva:
print("\nfull_day_res (H=60):\n")
display(full_day_res[60]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))
print("\nfull_day_res (H=90):\n")
display(full_day_res[90]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))


full_day_res (H=60):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,60,0,15,ema_60,True,max abs_IC_OOS,-0.193498,-0.189089,0.189089,229712,
15,60,1,1,roc_60,True,max abs_IC_OOS,-0.187085,-0.177769,0.177769,229712,



full_day_res (H=90):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
5,90,1,26,ema_60,True,max abs_IC_OOS,-0.245322,-0.238691,0.238691,229712,
31,90,2,1,roc_60,True,max abs_IC_OOS,-0.234815,-0.225123,0.225123,229712,
0,90,0,5,atr_norm_20,True,max abs_IC_OOS,0.123236,0.107830,0.107830,229712,


In [85]:
print("\ngestation_res (H=60):\n")
display(gest_res[60]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))
print("\ngestation_res (H=90):\n")
display(gest_res[90]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))


gestation_res (H=60):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,60,0,23,ema_60,True,max abs_IC_OOS,-0.354751,-0.448799,0.448799,35746,
30,60,5,1,roc_30,True,max abs_IC_OOS,-0.308784,-0.395103,0.395103,35746,
31,60,6,1,roc_60,True,max abs_IC_OOS,-0.318990,-0.383652,0.383652,35746,
29,60,4,1,roc_20,True,max abs_IC_OOS,-0.277609,-0.373835,0.373835,35746,
23,60,1,3,momentum_10,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.221862,-0.294512,0.294512,35746,
27,60,3,2,momentum_5,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.174024,-0.218387,0.218387,35746,
26,60,2,1,momentum_3,True,max abs_IC_OOS,-0.140744,-0.168190,0.168190,35746,



gestation_res (H=90):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,90,0,23,ema_60,True,max abs_IC_OOS,-0.287444,-0.320478,0.320478,35746,
31,90,6,1,roc_60,True,max abs_IC_OOS,-0.255514,-0.285494,0.285494,35746,
30,90,5,1,roc_30,True,max abs_IC_OOS,-0.251566,-0.283316,0.283316,35746,
29,90,4,1,roc_20,True,max abs_IC_OOS,-0.226251,-0.262057,0.262057,35746,
23,90,1,3,momentum_10,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.180712,-0.211260,0.211260,35746,
27,90,3,2,momentum_5,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.147928,-0.163546,0.163546,35746,
26,90,2,1,momentum_3,True,max abs_IC_OOS,-0.118749,-0.130582,0.130582,35746,


In [86]:
print("\nexecution_res (H=60):\n")
display(exe_res[60]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))
print("\nexecution_res (H=60):\n")
display(exe_res[90]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))


execution_res (H=60):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,60,0,27,ema_60,True,max abs_IC_OOS,-0.594627,-0.556101,0.556101,35746,
30,60,3,1,roc_20,True,max abs_IC_OOS,-0.522224,-0.491414,0.491414,35746,
31,60,4,1,roc_60,True,max abs_IC_OOS,-0.542549,-0.484781,0.484781,35746,
28,60,2,2,momentum_5,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.335179,-0.309279,0.309279,35746,
27,60,1,1,momentum_3,True,max abs_IC_OOS,-0.270427,-0.252210,0.252210,35746,



execution_res (H=60):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,90,0,27,ema_60,True,max abs_IC_OOS,-0.621965,-0.597670,0.597670,35746,
30,90,3,1,roc_20,True,max abs_IC_OOS,-0.543528,-0.526892,0.526892,35746,
31,90,4,1,roc_60,True,max abs_IC_OOS,-0.569973,-0.519390,0.519390,35746,
28,90,2,2,momentum_5,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.344905,-0.331683,0.331683,35746,
27,90,1,1,momentum_3,True,max abs_IC_OOS,-0.278034,-0.268643,0.268643,35746,


#### **4.3.4. Comentarios generales – Resolución de clusters**

**1. Lectura Global**

El análisis conjunto de IC OOS y clusters de correlación muestra tres conclusiones estructurales claras.

A. `ema_60` como factor dominante

- Aparece en todas las ventanas analizadas: jornada completa, gestación y expansión.
- Aparece en ambos horizontes (60 y 90 minutos).
- Siempre pertenece a clusters grandes (alta redundancia), pero:
  - presenta el mayor `|IC_OOS|`,
  - mantiene un IC negativo fuerte y estable en IS y OOS.

Conclusión:
> `ema_60` actúa como un factor estructural de tendencia intradía, robusto y no dependiente del horario.

B. Los indicadores `roc_*` como factores de magnitud

- `roc_60` aparece de forma consistente:
  - en la jornada completa (H=60 y H=90),
  - en la ventana de gestación,
  - en la ventana de expansión.
- `roc_20` y `roc_30` emergen únicamente en ventanas específicas (gestación y expansión).

Conclusión:
> La magnitud del movimiento reciente aporta información predictiva, pero la escala temporal óptima depende del tramo horario.

C. Los indicadores `momentum_*` como factores de aceleración

- No aparecen como relevantes en la jornada completa.
- Sí aparecen de forma consistente en las ventanas de gestación y expansión.
- Dominan escalas cortas (3, 5 y 10).
- Presentan IC OOS fuerte y estable dentro de esas ventanas.

Conclusión:
> El momentum de corto plazo no es un factor estructural del día completo, pero resulta altamente informativo en ventanas activas del mercado.

**2. Diferencias por ventana (lectura operativa)**

- Jornada completa (complete)

  - Conjunto de indicadores reducido:
    - `ema_60`
    - `roc_60`
    - `atr_norm_20` (solo para H=90)
  - Señales robustas, pero de naturaleza más lenta.
  - Adecuadas para:
    - definir estructura,
    - establecer baseline,
    - actuar como filtros de régimen.

- Ventana de gestación (08:00–09:00)

  - Incremento notable de señal:
    - `ema_60`
    - `roc_20` / `roc_30` / `roc_60`
    - `momentum_3` / 5 / 10
  - IC OOS elevado (aprox. 0.2–0.45).

  Conclusión:
  > Esta ventana concentra información direccional temprana relevante para anticipar movimientos posteriores.

- Ventana de expansión (09:00–10:00)

  - Señales aún más fuertes que en gestación.
  - IC OOS superior a 0.5 en varios factores.
  - Dominan:
    - tendencia (`ema_60`),
    - magnitud (`roc_*`),
    - aceleración (`momentum_*`).

  Conclusión:
  > Esta ventana resulta especialmente adecuada para decisiones operativas más agresivas.

Aclaraciones importantes

- Que los indicadores momentum_* no aparezcan en la jornada completa no invalida su utilidad.
- La presencia de IC negativos no es un problema; solo indica una relación inversa con el target.
- La existencia de clusters grandes es normal y esperable en indicadores técnicos altamente relacionados.

Conclusión general

> El análisis conjunto de IC fuera de muestra y clusters de correlación revela una estructura jerárquica clara:
> - un factor de tendencia estructural (ema_60),
> - factores de magnitud (roc_*) con dependencia temporal,
> - y factores de aceleración (momentum_*) relevantes únicamente en ventanas activas.
>
> Esta evidencia justifica una selección jerárquica de features, combinando un core estructural válido para toda la jornada con features específicas condicionadas por la ventana horaria.

In [87]:
print('TI full day finals H=60:', full_day_final_60)
print('TI full day finals H=90:', full_day_final_90)

print('TI gestation finals H=60:', gestation_final_60)
print('TI gestation finals H=90:', gestation_final_90)

print('TI execution finals H=60:', execution_final_60)
print('TI execution finals H=90:', execution_final_90)

TI full day finals H=60: ['ema_60', 'roc_60']
TI full day finals H=90: ['atr_norm_20', 'ema_60', 'roc_60']
TI gestation finals H=60: ['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']
TI gestation finals H=90: ['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']
TI execution finals H=60: ['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60']
TI execution finals H=90: ['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60']


### **4.4. Correlación final**

Una vez reducido el conjunto de indicadores técnicos mediante el filtrado por fuerza de señal (|IC| ≥ 0.10) y la resolución por clusters, analizaremos la matriz de correlación entre los indicadores resultantes con el objetivo de evaluar posibles redundancias adicionales.

**Criterio final para el análisis de correlación entre indicadores técnicos**

En esta etapa, el criterio de selección deja de ser puramente estadístico y pasa a ser arquitectónico.  
El análisis de Information Coefficient (IC) y los clusters de correlación ya cumplieron su función principal: identificar señal y eliminar redundancia evidente.  
La correlación final no se utiliza para “volver a filtrar” indicadores, sino para ordenar su uso dentro del modelo.

**Principio rector**

> No todos los indicadores finales deben competir entre sí:  
> deben coexistir por rol funcional y por ventana horaria.

**Definición de roles**

Antes de analizar correlaciones, los indicadores se agrupan por el fenómeno de mercado que representan:

  - Estructura / tendencia  
    `ema_60`

  - Magnitud del movimiento  
    `roc_60`, `roc_20`, `roc_30`

  - Aceleración / impulso  
    `momentum_3`, `momentum_5`, `momentum_10`

  - Riesgo / volatilidad  
    `atr_norm_20`

La correlación solo debe penalizar redundancia dentro de un mismo rol, no entre roles distintos.

**Criterio de uso de la correlación**

A. Dentro de un mismo rol  

  La correlación se utiliza para reducir redundancia:

  - |ρ| ≥ 0.85 se considera redundancia fuerte.
  - Se conserva:
    - el indicador con mayor |IC_OOS|, o
    - el más estable entre ventanas, si los IC son similares.

  Ejemplo:
  - momentum_3, momentum_5 y momentum_10 suelen estar altamente correlacionados.
  - Se conserva uno solo por ventana.

B. Entre roles distintos

  La correlación no se utiliza como criterio de descarte, salvo casos extremos.

  Ejemplos:
  - ema_60 vs roc_60  
  - roc_60 vs momentum_5  

  Aunque puedan mostrar correlación, capturan fenómenos distintos (tendencia, magnitud, aceleración) y pueden ser explotados conjuntamente por modelos no lineales.

**Aplicación directa a los indicadores finales**

- Core estructural (invariable)

  Estos indicadores se conservan siempre:

  - `ema_60`
  - `roc_60`

  Son estables, robustos fuera de muestra, válidos en múltiples horizontes y ventanas.

- Jornada completa

  - H = 60: `ema_60`, `roc_60`
  - H = 90: `atr_norm_20`, `ema_60`, `roc_60`

  El indicador de volatilidad (`atr_norm_20`) cumple un rol distinto y no compite con EMA o ROC.

- Ventanas de gestación y expansión

  En estas ventanas se aplica correlación intra-rol:

  - Momentum:
    - conservar un único momentum corto (por ejemplo, momentum_5 o momentum_3).
  - ROC:
    - conservar roc_60 como base y, opcionalmente, un ROC de escala más corta.

  Ejemplo razonable por ventana:
  - ema_60
  - roc_60
  - roc_20 o roc_30
  - momentum_5

**Regla operativa final**

> La correlación final no se utiliza para eliminar señales válidas,  
> sino para evitar redundancia dentro del mismo rol funcional.  
> Los indicadores estructurales (ema_60, roc_60) se conservan siempre,  
> mientras que los indicadores dependientes de ventana se seleccionan maximizando diversidad funcional: tendencia, magnitud, aceleración y riesgo.

**Síntesis práctica**

- Core fijo: ema_60, roc_60  
- Por ventana:
  - un ROC corto,
  - un momentum corto,
  - opcionalmente un indicador de volatilidad.

A partir de aquí, la responsabilidad de combinar y ponderar estas señales se delega al modelo.


### **4.5. Tratamiento de familias en ventanas de gestación y expansión**

In [91]:
import pandas as pd

def pick_one_by_family(
    *,
    window_name: str,
    horizon: int,
    finals_list: list[str],
    ic_table_window: pd.DataFrame,     # ic_table_full_day / ic_table_gestation / ic_table_execution
    corr_mean_window: pd.DataFrame,    # corr_mean correspondiente (misma ventana y horizonte)
    families: dict[str, list[str]],    # grupos candidatos redundantes (momentum, roc_short, etc.)
    corr_threshold: float = 0.85,
    score_col: str = "abs_IC_OOS",     # criterio principal: señal OOS
) -> dict:
    """
    Para una ventana/horizonte:
      - Mantiene el core que no está en families (o si lo incluye, lo respeta según usted defina)
      - Dentro de cada familia:
          - si los candidatos están altamente correlacionados (|rho|>=threshold),
            se queda con 1: el de mayor score_col (abs_IC_OOS)
          - si NO están altamente correlacionados entre sí, los deja (no los fuerza a 1)
    Devuelve:
      - selected: lista final
      - decisions: tabla con decisiones por familia
    """

    # -------------------------
    # 1) Tomar IC solo de este horizonte
    # -------------------------
    ic_h = ic_table_window[ic_table_window["horizon"] == horizon].copy()

    # Si no existe abs_IC_OOS, lo creamos
    if score_col not in ic_h.columns and "IC_OOS" in ic_h.columns:
        ic_h[score_col] = ic_h["IC_OOS"].abs()

    ic_h = ic_h.set_index("indicator")

    # -------------------------
    # 2) Sanity: aseguramos que corr tenga índices/cols compatibles
    # -------------------------
    corr = corr_mean_window.copy()
    corr = corr.loc[corr.index, corr.index]  # alinea si hace falta

    finals_set = set(finals_list)

    selected = set(finals_list)  # empezamos manteniendo todo
    decision_rows = []

    # -------------------------
    # 3) Resolver familia por familia
    # -------------------------
    for fam_name, fam_members in families.items():

        # Solo considerar los miembros que realmente están en esta lista final
        cand = [x for x in fam_members if x in finals_set]

        # Si hay 0 o 1, no hay nada que resolver
        if len(cand) <= 1:
            if len(cand) == 1:
                decision_rows.append({
                    "window": window_name, "horizon": horizon, "family": fam_name,
                    "candidates": cand, "action": "keep (single)", "kept": cand[0],
                    "reason": "only one candidate present"
                })
            continue

        # -------------------------
        # 3a) Ver si son redundantes (alta correlación)
        #     Criterio simple: si el máximo |rho| entre pares >= threshold, consideramos redundancia
        # -------------------------
        max_abs_rho = 0.0
        for i in range(len(cand)):
            for j in range(i + 1, len(cand)):
                if cand[i] in corr.index and cand[j] in corr.columns:
                    rho = corr.at[cand[i], cand[j]]
                    if pd.notna(rho):
                        max_abs_rho = max(max_abs_rho, abs(rho))

        # Si NO hay redundancia fuerte, los dejamos todos
        if max_abs_rho < corr_threshold:
            decision_rows.append({
                "window": window_name, "horizon": horizon, "family": fam_name,
                "candidates": cand, "action": "keep (not redundant)", "kept": cand,
                "reason": f"max |rho|={max_abs_rho:.3f} < {corr_threshold}"
            })
            continue

        # -------------------------
        # 3b) Redundantes: elegir el de mayor señal OOS (abs_IC_OOS)
        # -------------------------
        # Tomamos el score de cada candidato; si falta, lo dejamos en -inf para no elegirlo
        scored = []
        for ind in cand:
            score = ic_h.at[ind, score_col] if ind in ic_h.index and pd.notna(ic_h.at[ind, score_col]) else float("-inf")
            scored.append((ind, score))

        scored.sort(key=lambda t: (-t[1], t[0]))  # mayor score, desempate alfabético
        kept, kept_score = scored[0]

        # Eliminamos los demás de la selección
        for ind, _ in scored[1:]:
            if ind in selected:
                selected.remove(ind)

        decision_rows.append({
            "window": window_name, "horizon": horizon, "family": fam_name,
            "candidates": cand, "action": "select_one", "kept": kept,
            "reason": f"redundant (max |rho|={max_abs_rho:.3f} >= {corr_threshold}); kept by max {score_col}={kept_score:.6f}"
        })

    decisions = pd.DataFrame(decision_rows)
    return {"selected": sorted(selected), "decisions": decisions}


In [93]:
# Familias redundantes que queremos resolver dentro de cada ventana
FAMILIES = {
    "momentum":   ["momentum_3", "momentum_5", "momentum_10"],
    "roc_short":  ["roc_20", "roc_30"],  # roc_60 es core, no lo metemos aquí
}

# ---- Gestation ----
gest60 = pick_one_by_family(
    window_name="gestation",
    horizon=60,
    finals_list=['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60'],
    ic_table_window=ic_table_gestation,
    corr_mean_window=corr_mean_ic_60_relevant_gestation,
    families=FAMILIES,
    corr_threshold=0.85,
)

gest90 = pick_one_by_family(
    window_name="gestation",
    horizon=90,
    finals_list=['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60'],
    ic_table_window=ic_table_gestation,
    corr_mean_window=corr_mean_ic_90_relevant_gestation,
    families=FAMILIES,
    corr_threshold=0.85,
)

print("Gestation H=60 selected:", gest60["selected"])
#display(gest60["decisions"])

print("Gestation H=90 selected:", gest90["selected"])
#display(gest90["decisions"])


# ---- Expansion ----
exp60 = pick_one_by_family(
    window_name="execution",
    horizon=60,
    finals_list=['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60'],
    ic_table_window=ic_table_execution,
    corr_mean_window=corr_mean_ic_60_relevant_execution,
    families=FAMILIES,
    corr_threshold=0.85,
)

exp90 = pick_one_by_family(
    window_name="execution",
    horizon=90,
    finals_list=['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60'],
    ic_table_window=ic_table_execution,
    corr_mean_window=corr_mean_ic_90_relevant_execution,
    families=FAMILIES,
    corr_threshold=0.85,
)

print("Execution H=60 selected:", exp60["selected"])
#display(exp60["decisions"])

print("Execution H=90 selected:", exp90["selected"])
#display(exp90["decisions"])



Gestation H=60 selected: ['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']
Gestation H=90 selected: ['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']
Execution H=60 selected: ['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60']
Execution H=90 selected: ['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60']


**Conclusión sobre la selección final por correlación en ventanas activas**

Los resultados del análisis muestran que no existe redundancia fuerte dentro de las familias de indicadores evaluadas bajo el umbral definido.

En particular:

- Para los indicadores de momentum (3, 5 y 10):
  - el valor máximo de correlación absoluta se encuentra en el rango |ρ| ≈ 0.71–0.72,
  - dicho valor es inferior al umbral de redundancia establecido (0.85).

- Para los ROC de corto plazo (roc_20 y roc_30):
  - el valor máximo de correlación absoluta es |ρ| ≈ 0.69,
  - nuevamente por debajo del umbral de redundancia.

Bajo este criterio, los indicadores **no son considerados redundantes**, y por lo tanto el algoritmo decide correctamente no forzar la selección de un único representante dentro de cada familia. Esto se refleja en la decisión `keep (not redundant)`.

**Interpretación conceptual**

Aunque los indicadores pertenecen a la misma familia conceptual, no describen exactamente el mismo fenómeno:

- En el caso del momentum, cada escala captura una dinámica distinta de aceleración del precio.
- En el caso del ROC corto, distintas longitudes de ventana capturan magnitudes del movimiento que no evolucionan de forma idéntica.

Esto indica que, durante las ventanas de gestación y expansión, el mercado presenta una **estructura multiescala real**, donde múltiples escalas aportan información complementaria.

**Por qué este resultado es correcto**

Este comportamiento no implica:
- un error en el criterio de selección,
- un exceso injustificado de features,
- ni un riesgo inmediato de confusión para el modelo.

Por el contrario, indica que el pipeline:
- detecta correctamente la ausencia de redundancia fuerte,
- evita colapsar artificialmente señales distintas,
- y respeta la evidencia empírica observada en los datos.

**Cuándo sería razonable forzar una reducción**

La selección de un único indicador por familia solo estaría justificada si:
- la correlación absoluta fuera muy elevada (|ρ| ≥ 0.85–0.90), y
- los indicadores presentaran un IC OOS similar.

Ese no es el caso en este análisis.

**Decisión recomendada**

Bajo el criterio actual, la decisión más consistente es:
- mantener múltiples escalas de momentum y ROC corto en ventanas activas,
- y delegar en el modelo (regularización, árboles, mecanismos de atención) la ponderación efectiva de cada señal.

En síntesis:

> En las ventanas activas, momentum y ROC corto aportan información complementaria de naturaleza multiescala,  
> por lo que no deben ser forzados a un único indicador bajo el criterio de correlación actual.


### **4.4. Decisión de consolidación final**

A) Estrategia general de entrenamiento

A partir del análisis de Information Coefficient (IC), correlación y clusters de redundancia, se concluye que **no es necesario entrenar modelos distintos por ventana horaria**.  
Es metodológicamente más sólido entrenar **un único modelo** que incorpore:

a. Un **core de features estructurales**, activas durante toda la jornada.  
b. Un conjunto de **features adicionales activadas de forma condicional** durante las ventanas de gestación y expansión.

Este enfoque permite:
- utilizar todo el volumen de datos disponible,
- evitar la fragmentación del dataset,
- reducir el riesgo de sobreajuste,
- respetar la evidencia empírica que muestra que ciertas señales solo son informativas en ventanas específicas.

El modelo aprende de forma implícita **cuándo** una feature aporta señal y cuándo no, sin forzar reglas externas ni entrenamientos separados.

---

B) Consolidación del listado final de features

Bajo este criterio, el conjunto final de features queda definido de la siguiente manera.

B.1) Target delta_60

Todo el día:
- ema_60  
- roc_60  

Ventana de gestación:
- ema_60  
- roc_60  
- momentum_10  
- momentum_3  
- momentum_5  
- roc_20  
- roc_30  

Ventana de expansión:
- ema_60  
- roc_60  
- momentum_3  
- momentum_5  
- roc_20  

---

B.2) Target delta_90

Todo el día:
- ema_60  
- roc_60  
- atr_norm_20  

Ventana de gestación:
- ema_60  
- roc_60  
- momentum_10  
- momentum_3  
- momentum_5  
- roc_20  
- roc_30  

Ventana de expansión:
- ema_60  
- roc_60  
- momentum_3  
- momentum_5  
- roc_20  

---

C) Evaluación del criterio adoptado

Este esquema es coherente con todos los resultados obtenidos:

a. ema_60 y roc_60 actúan como **factores estructurales**, robustos a lo largo de toda la jornada.  
b. atr_norm_20 aporta información de **riesgo y volatilidad**, relevante principalmente para horizontes más largos (delta_90).  
c. Los indicadores momentum_* y roc_* de corto plazo muestran **capacidad predictiva dependiente del horario**, especialmente en ventanas activas.  
d. No se observa redundancia fuerte que justifique eliminar estas señales dentro de las ventanas definidas.

---

D) Conclusión final

Es adecuado entrenar un único modelo con un core estructural activo durante todo el día y features adicionales activadas de forma condicional por ventana horaria.  
El listado final de features propuesto es consistente con el análisis empírico realizado y representa un equilibrio correcto entre robustez global y sensibilidad temporal.


In [ ]:
# ============================================================
# Consolidación final de indicadores técnicos
# ============================================================

# Se combinan todos los indicadores seleccionados en:
# - jornada completa
# - ventana de gestación
# - ventana de expansión
# - para ambos horizontes (H=60 y H=90)

tech_indicators_finals = sorted(
    set(
        complete_final_60
        + complete_final_90
        + gestation_final_60
        + gestation_final_90
        + expansion_final_60
        + expansion_final_90
    )
)

# Mostrar resultado final
print("tech_indicators_finals:")
print(tech_indicators_finals)

## **5. Análisis de coherencia direccional**

Una vez consolidado el conjunto final de indicadores técnicos:

```
technical_indicators_finals = [
    "price_ema60",
    "momentum_10",
    "roc_30",
    "roc_60",
]
```
el siguiente paso consiste en evaluar la coherencia direccional de cada indicador, contrastando dos dimensiones complementarias:

- IC_delta: relación del indicador con la magnitud del movimiento futuro.

- IC_trade_only: relación del indicador con la dirección del movimiento en instantes donde existe trade.

Este análisis permite responder una pregunta clave:

    ¿El indicador señala la misma dirección cuando explica magnitud del movimiento y cuando explica dirección operativa?

**Criterio de coherencia**

Un indicador se considera coherente si:

- `sign(IC_delta)` == `sign(IC_trade_only)`
- ninguno de los dos es 0 o NaN

Adicionalmente, se analiza la relación de intensidades:

- si `|IC_trade_only|` > `|IC_delta|`, el indicador refuerza su señal en contextos de decisión operativa, indicando mayor utilidad práctica.

Este criterio permite validar no solo la significancia estadística, sino también la consistencia económica y operativa de las señales seleccionadas.

### **5.1. Código para análisis de coherencia direccional**

In [ ]:
import numpy as np
import pandas as pd

def check_directional_coherence(df_selected: pd.DataFrame) -> pd.DataFrame:
    """
    Verifica coherencia direccional entre IC_delta e IC_trade_only.

    Criterio:
      - coherent = True si sign(IC_delta) == sign(IC_trade_only) y ninguno es 0/NaN
      - coherent = False en caso contrario

    Nota:
      - La columna 'family' es opcional. Si no existe, se omite del output.
    """
    out = df_selected.copy()

    # Signos (NaN -> NaN)
    out["sign_IC_delta"] = np.sign(out["IC_delta"])
    out["sign_IC_trade_only"] = np.sign(out["IC_trade_only"])

    # Coherencia: mismos signos y ambos distintos de 0 y no NaN
    out["coherent"] = (
        out["IC_delta"].notna()
        & out["IC_trade_only"].notna()
        & (out["sign_IC_delta"] != 0)
        & (out["sign_IC_trade_only"] != 0)
        & (out["sign_IC_delta"] == out["sign_IC_trade_only"])
    )

    # Comparación de magnitudes
    out["abs_IC_trade_only"] = out["IC_trade_only"].abs()
    out["abs_IC_delta"] = out["IC_delta"].abs()  # por si no estuviera
    out["abs_gap_trade_vs_delta"] = out["abs_IC_trade_only"] - out["abs_IC_delta"]

    # Orden
    out = out.sort_values(["coherent", "abs_IC_delta"], ascending=[True, False])

    # Columnas base a devolver
    cols = [
        "indicator", "IC_delta", "IC_trade_only",
        "sign_IC_delta", "sign_IC_trade_only", "coherent",
        "abs_IC_delta", "abs_IC_trade_only", "abs_gap_trade_vs_delta"
    ]

    # Si existe 'family', la incluimos al inicio
    if "family" in out.columns:
        cols = ["family"] + cols

    return out[cols]

### **5.2. Aplicación**

In [ ]:
ic_selected_60 = ic_table[
    (ic_table["horizon"] == 60) &
    (ic_table["indicator"].isin(technical_indicators_finals))
].copy()

coherence_60 = check_directional_coherence(ic_selected_60)

In [ ]:
ic_selected_90 = ic_table[
    (ic_table["horizon"] == 90) &
    (ic_table["indicator"].isin(technical_indicators_finals))
].copy()

coherence_90 = check_directional_coherence(ic_selected_90)


### **5.3. Resultados y análisis**

In [ ]:
coherence_60

In [ ]:
coherence_90

El análisis de coherencia direccional se aplicó sobre el conjunto final de indicadores técnicos seleccionados (`price_ema60`, `momentum_10`, `roc_30`, `roc_60`) para los horizontes de 60 y 90 minutos, comparando la señal asociada a la magnitud del movimiento (`IC_delta`) y a la dirección operativa cuando hay trade (`IC_trade_only`).

Resultados principales

- Coherencia perfecta en ambos horizontes

  En todos los casos se cumple que `sign(IC_delta) = sign(IC_trade_only)` y ambos son distintos de cero.
  Esto indica que los indicadores no mezclan regímenes y mantienen una interpretación direccional consistente entre magnitud y decisión operativa.

- Régimen dominante de reversión

  Todos los indicadores presentan valores negativos tanto en `IC_delta` como en `IC_trade_only`.
  Esto confirma que, durante la ventana de gestación, las extensiones del precio tienden a anticipar agotamiento o reversión en los horizontes analizados (60 y 90 minutos).

- Señal más fuerte en contexto operativo

  En todos los indicadores se observa que $$∣𝐼𝐶_{𝑡𝑟𝑎𝑑𝑒-only}| >  ∣𝐼𝐶_{\Delta}| $$
  
  lo que implica que la señal es más intensa cuando existe una oportunidad de trade, reforzando su utilidad práctica.

- Estabilidad inter-horizonte

  El patrón observado en 60 minutos se replica de forma consistente en 90 minutos, tanto en signo como en magnitudes relativas.
  Esto sugiere que el set de indicadores captura una estructura temporal robusta, no dependiente de un único horizonte.




### **5.4. Conclusión operativa**



  El conjunto de indicadores técnicos seleccionado presenta coherencia direccional total, estabilidad entre horizontes y una señal más fuerte en contextos operativos reales.
  
  Esto valida su uso como núcleo definitivo de features para el modelo, sin ambigüedad de régimen y con interpretación económica clara.

In [ ]:
technical_indicators_finals

## **6. Análisis de OHLCV**

Las variables OHLCV (`open`, `high`, `low`, `close`, `volume`) representan el estado instantáneo del mercado y constituyen la materia prima a partir de la cual se construyen los indicadores técnicos.

Aunque no están diseñadas explícitamente como señales anticipatorias, su inclusión directa en el modelo puede:

- aportar información contextual relevante,
- introducir redundancia respecto a los indicadores derivados,
- o, en algunos casos, mostrar capacidad predictiva directa.

Por este motivo, se realiza un análisis específico de Information Coefficient (IC) y correlación para las variables OHLCV, con el objetivo de evaluar su contribución real y justificar su inclusión en el modelo.

### **6.1. IC de variables OHLCV vs targets**


Se evalúa la relación entre OHLCV y los targets definidos (`delta_pts_h`, `trade_h`, etc.) en la ventana de gestación, utilizando el mismo criterio metodológico aplicado a los indicadores técnicos.


In [ ]:
mnq_intraday_with_indicators.head()

In [ ]:
"""
    Calcula una tabla resumen de Information Coefficient (IC) para cada
    indicador técnico y cada horizonte temporal (60 / 90).

    Para cada indicador y horizonte se evalúa su relación con:
    - delta_pts_h        → magnitud del movimiento futuro (continuo)
    - trade_h            → dirección (-1, 0, +1)
    - trade_h_only       → dirección pura (excluye no-trade)
    - target_op_h        → evento operativo (0 / 1)
    - target_tail_h      → evento extremo (0 / 1)

    Parámetro clave:
    - use_daily_ic = True
        Calcula IC por día y luego promedia (más robusto estadísticamente)
    - use_daily_ic = False
        Calcula IC global usando todas las filas juntas
    """


In [ ]:
ohlcv_cols = ["open", "high", "low", "close", "volume"]


In [ ]:
ic_table_ohlcv = compute_ic_table_is_oos_delta(
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    horizons=(60, 90),
    window_start=start_time_indicators,
    window_end=final_time_indicators,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

In [ ]:
ic_table_ohlcv

#### **6.1.1. Análisis de resultados de IC**

**1. OHLC presentan señal fuerte y coherente**

Las variables de precio (`open`, `high`, `low`, `close`) exhiben valores de:

$$
|IC_{\Delta}| \approx 0.30 \;-\; 0.44
$$

con **signo negativo consistente** tanto en `IC_delta` como en `IC_trade_only`.

Esto indica que:

- El **nivel de precio en la ventana de gestación** contiene información relevante
  sobre la magnitud y dirección del movimiento futuro.
- El régimen dominante es de **reversión**, consistente con lo observado
  previamente en los indicadores técnicos.
- La señal es incluso **más intensa cuando hay trade**, dado que:

$$
|IC_{trade\_only}| > |IC_{\Delta}|
$$

En términos puramente estadísticos, las variables OHLC muestran una capacidad
predictiva **comparable en magnitud** a la de los mejores indicadores técnicos.

<br>

**2. El volumen no presenta señal directa relevante**

La variable `volume` muestra:

$$
|IC_{\Delta}| \approx 0.01 \;-\; 0.015
$$

con valores cercanos a cero y sin coherencia clara entre los distintos targets.

Esto indica que, en la ventana de gestación analizada:

- El volumen **no explica directamente** la magnitud del movimiento futuro.
- Su aporte como señal anticipatoria es **débil o nulo**.





#### **6.1.2. Interpretación metodológica**

Aunque las variables OHLC presentan valores de IC elevados, este resultado
**no invalida** el rol central de los indicadores técnicos. Por el contrario,
confirma que:

- Los indicadores técnicos capturan y **reexpresan información ya contenida
  en el precio**, de forma más estructurada y filtrada.
- Las variables OHLC actúan como **variables de estado del mercado**, no como
  señales diseñadas explícitamente para anticipar movimientos.

En este contexto, el IC elevado de OHLC es esperable y no implica que deban
sustituir a los indicadores técnicos, sino que:

- **Refuerzan la consistencia del análisis**, al confirmar el régimen dominante.
- Aportan **contexto estructural** útil para el modelo.

#### **6.1.3. Decisión de diseño del modelo**


En base a este análisis, se adopta el siguiente criterio:

- Las variables **OHLC** se mantienen como **features base**, normalizadas o
  transformadas según corresponda.
- Los **indicadores técnicos seleccionados** constituyen el **núcleo predictivo**
  del modelo.
- La variable **`volume` no se utiliza como feature principal** en esta etapa,
  pudiendo reservarse para análisis contextuales futuros.

Esta decisión permite equilibrar **información estructural**, **capacidad
predictiva real** e **interpretabilidad económica**, evitando tanto redundancia
innecesaria como exclusiones arbitrarias.

### **6.2. Correlación entre OHLCV e indicadores técnicos seleccionados**


El objetivo de este paso es evaluar redundancia informativa, es decir, determinar si las variables OHLCV ya están implícitamente capturadas por los indicadores técnicos seleccionados.

In [ ]:
technical_indicators_finals

In [ ]:
corr_ohlcv_vs_indicators = (
    mnq_intraday_with_indicators[
        ohlcv_cols + technical_indicators_finals
    ]
    .corr(method="spearman")
    .loc[ohlcv_cols, technical_indicators_finals]
)

corr_ohlcv_vs_indicators

#### **6.2.1. Análisis de Correlación entre variables OHLCV e indicadores técnicos seleccionados**

Se analizó la correlación de Spearman entre las variables OHLCV (`open`, `high`,
`low`, `close`, `volume`) y el conjunto final de indicadores técnicos seleccionados
(`price_ema60`, `momentum_10`, `roc_30`, `roc_60`), con el objetivo de evaluar
posible redundancia informativa.

**1. OHLC y los indicadores técnicos no son redundantes**

Las variables de precio (`open`, `high`, `low`, `close`) presentan correlaciones
muy cercanas a cero respecto a todos los indicadores técnicos:

$$
|\rho| \approx 0.00 \;-\; 0.01
$$

Esto indica que:

- Las variables OHLC **no están linealmente correlacionadas** con los indicadores.
- No existe colinealidad directa entre el estado del precio y las señales técnicas.
- Aunque ambos grupos tengan IC elevado respecto al target, **aportan información
  desde perspectivas distintas**.

<br>

**2. El volumen muestra correlación débil y no estructural**

La variable `volume` presenta correlaciones ligeramente mayores en magnitud:

$$
|\rho| \approx 0.03 \;-\; 0.07
$$

Aun así:

- Las correlaciones siguen siendo bajas.
- No indican dependencia fuerte con los indicadores técnicos.
- En combinación con su bajo IC, se refuerza que el volumen **no aporta señal
  predictiva directa** en esta etapa.

#### **6.2.2. Conclusión metodológica**


El análisis cruzado confirma que:

- Las variables **OHLC** aportan información de **estado del mercado**.
- Los **indicadores técnicos** capturan **estructura, dinámica y régimen**.
- Ambos conjuntos son **complementarios**, no redundantes.

Por lo tanto, se justifica incluir OHLC e indicadores técnicos seleccionados como
features del modelo, sin riesgo de colinealidad ni duplicación innecesaria de
información.


## **7. Consolidación de features**


A partir del análisis estadístico y estructural realizado en este stage, se concluye
que el conjunto de features adecuado para el entrenamiento del modelo está compuesto por:

- **Variables OHLC**: `open`, `high`, `low`, `close`
- **Indicadores técnicos seleccionados**:
  - `price_ema60`
  - `momentum_10`
  - `roc_30`
  - `roc_60`



In [ ]:
technical_indicators_finals = ["price_ema60", "momentum_10",  "roc_30", "roc_60"]

### **7.1. Justificación**


1. Fuerza predictiva comprobada

    Tanto las variables OHLC como los indicadores técnicos seleccionados presentan valores de Information Coefficient elevados frente a los targets definidos, con régimen direccional consistente en los horizontes de 60 y 90 minutos.

2. Coherencia direccional

    Los indicadores técnicos seleccionados muestran coherencia total entre IC_delta y IC_trade_only, confirmando estabilidad de régimen e interpretación económica clara.

3. Ausencia de redundancia crítica

    El análisis de correlación cruzada demuestra que:

    - Las variables OHLC no están linealmente correlacionadas con los indicadores técnicos seleccionados.
    - Los indicadores técnicos capturan información estructural y dinámica que no es una reexpresión directa del precio crudo.

4. Complementariedad informativa

    - OHLC aporta el estado del mercado.
    - Los indicadores técnicos aportan estructura, momentum y régimen.

    Ambos conjuntos son complementarios y necesarios para describir el proceso generador del movimiento intradía.

### **7.2. Decisión final y set consolidado de features**


A partir del análisis estadístico y estructural realizado, se decide entrenar el modelo utilizando exclusivamente el siguiente conjunto de features:

```
finals_features = {
    "open", "high", "low", "close",
    "price_ema60",
    "momentum_10",
    "roc_30", "roc_60",
}
```
Este set constituye un conjunto consolidado, válido tanto para horizontes de 60 como de 90 minutos, dado que las diferencias observadas entre ambos son cuantitativas y no cualitativas.

| Tipo de feature            | Variable                            | Rol económico principal                          |
|----------------------------|-------------------------------------|--------------------------------------------------|
| Precio intradía            | `open`, `high`, `low`, `close`      | Estado instantáneo y rango del precio            |
| Tendencia de precio        | `price_ema60`                       | Nivel tendencial y extensión del movimiento      |
| Momentum de extensión      | `momentum_10`                       | Intensidad del desplazamiento reciente           |
| Momentum direccional       | `roc_30`, `roc_60`                  | Velocidad y dirección del movimiento             |

Este conjunto combina información estructural (precio) con señales dinámicas (tendencia y momentum), ofreciendo un equilibrio óptimo entre capacidad predictiva, interpretabilidad económica y robustez estadística, y evitando tanto la redundancia innecesaria como la inclusión de ruido no informativo.

Nota: el componente de momentum direccional puede adaptarse al horizonte de predicción (`roc_60` para 60 min y `roc_30` para 90 min) sin modificar la arquitectura ni la lógica económica del modelo.

### **7.4. Implicancia para el pipeline**


Este set consolidado constituye el núcleo de features técnicas del modelo y será utilizado en las etapas posteriores para:

- entrenamiento y validación del modelo predictivo,
- análisis de contribución por feature,
- y evaluación de desempeño económico.

La consolidación reduce la dimensionalidad, mejora la interpretabilidad y alinea el diseño del modelo con la estructura temporal y económica observada en los datos.

### **4.3. Código para calcular indicadores técnicos consolidados**


In [ ]:
mnq_intraday_labeled = load_data()

In [ ]:
mnq_intraday_labeled

In [ ]:
from typing import List, Tuple
import pandas as pd
import ta
from ta.volatility import BollingerBands
from ta.momentum import ROCIndicator


def compute_selected_indicators_per_day(
    df: pd.DataFrame,
    target_col: str = "close",
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Computes selected technical indicators WITHOUT crossing days.
    All indicators are calculated independently per trading day
    using groupby('date').

    Indicators computed:
      - price_ema60
      - bb_60_15
      - rsi_14
      - roc_30
      - roc_60
      - macd

    Returns
    -------
    df_out : pd.DataFrame
        DataFrame with technical indicators added.
    indicator_columns : list[str]
        List of generated indicator column names.
    """

    indicator_columns: List[str] = [
        "price_ema60",
        "momentum_10",
        "roc_30",
        "roc_60",
          ]

    def apply_per_day(day_df: pd.DataFrame) -> pd.DataFrame:
        day_df = day_df.copy()

        # EMA-based price extension (normalized)
        day_df["price_ema60"] = (
            day_df[target_col] / day_df[target_col].ewm(span=60).mean() - 1
        )

        # Momentum (percentage change)
        day_df["momentum_10"] = day_df[target_col].pct_change(10)

        # Rate of Change (30, 60)
        day_df["roc_30"] = ROCIndicator(
            close=day_df[target_col],
            window=30
        ).roc()

        day_df["roc_60"] = ROCIndicator(
            close=day_df[target_col],
            window=60
        ).roc()


        return day_df

    df_out = df.groupby("date", group_keys=False).apply(apply_per_day)

    return df_out, indicator_columns



In [ ]:
mnq_technical_indicators, technical_indicators_features = compute_selected_indicators_per_day(mnq_intraday_labeled)

In [ ]:
ohlc_features = ['open', 'high', 'low', 'close']
final_features = ohlc_features + technical_indicators_finals
final_targets = ['delta_pts_60', 'delta_pts_90']
auxiliary_col = ['date']

In [ ]:
selected_columns = auxiliary_col + final_features + final_targets
mnq_features_targets = mnq_technical_indicators[selected_columns].copy()

In [ ]:
mnq_features_targets

In [ ]:
info_dataset(mnq_features_targets)

# Eliminar filas con al menos un NaN
mnq_features_targets = mnq_features_targets.dropna()

info_dataset(mnq_features_targets)


In [ ]:
OUT_PARQUET

In [ ]:
mnq_features_targets

In [ ]:
# Asegurar que exista el directorio
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

# Guardar dataset
mnq_features_targets.to_parquet(OUT_PARQUET, index=True)

## **8. Machine Learning for Algorithmic Trading**

### **8.1. Convertir indicadores en features estadísticamente estables**

Hasta ahora los tratamos como series crudas.

El siguiente nivel es normalizarlos en contexto intradía:

- Z-score rolling por día
- Percentil intradía
- Distancia al régimen típico del día

Ejemplo conceptual:
- roc_30 no vale por su valor absoluto
- vale por qué tan extremo es respecto a su distribución intradía

Esto aumenta IC out-of-sample sin cambiar indicadores.


In [ ]:
final_features

In [ ]:
technical_indicators_finals #Lo que va a mostrar ['price_ema60', 'momentum_10', 'roc_30', 'roc_60']
final_targets  #Lo que va a mostrar ['delta_pts_60', 'delta_pts_90']

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Configuración (usa tus listas ya definidas)
# ============================================================
# technical_indicators_finals = ['price_ema60', 'momentum_10', 'roc_30', 'roc_60']
# final_targets = ['delta_pts_60', 'delta_pts_90']

EPS = 1e-12  # Para evitar división por cero en la std


# ============================================================
# Función: Z-score "point-in-time" por día (sin fuga)
# ============================================================
def add_expanding_zscore_by_day(
    df: pd.DataFrame,
    cols: list[str],
    date_col: str = "date",
    min_periods: int = 10,
) -> pd.DataFrame:
    """
    Crea features normalizadas por día usando estadísticos "hasta el momento".

    Para cada día (date_col), y para cada minuto t dentro de ese día:
        z(t) = (x(t) - mean(x[<=t])) / std(x[<=t])

    Ventajas:
    - No mezcla días (cada día se normaliza por separado)
    - No usa información del futuro (solo datos hasta t)
    - Suele mejorar estabilidad OOS cuando hay cambios de régimen/volatilidad

    Parámetros:
    - cols: columnas a normalizar
    - date_col: columna que identifica el día (por ejemplo, 'date')
    - min_periods: mínimos puntos del día para empezar a calcular mean/std
                  (antes de eso devuelve NaN en el z-score)
    """
    out = df.copy()

    # Validaciones mínimas
    if not isinstance(out.index, pd.DatetimeIndex):
        raise TypeError("El índice debe ser DatetimeIndex (datetime).")
    if date_col not in out.columns:
        raise KeyError(f"Falta la columna '{date_col}' para agrupar por día.")

    # Asegura orden temporal global (por seguridad)
    out = out.sort_index()

    # Agrupa por día (no reordena los grupos)
    g = out.groupby(date_col, sort=False)

    # Calcula z-score expanding por día para cada feature
    for c in cols:
        exp_mean = (
            g[c]
            .expanding(min_periods=min_periods)
            .mean()
            .reset_index(level=0, drop=True)
        )
        exp_std = (
            g[c]
            .expanding(min_periods=min_periods)
            .std(ddof=0)
            .reset_index(level=0, drop=True)
        )

        # Feature normalizada
        out[f"{c}_z_exp"] = (out[c] - exp_mean) / (exp_std + EPS)

    return out


# ============================================================
# Pipeline: crea raw + z_exp para tus indicadores finales
# ============================================================
df = mnq_features_targets.copy()

# 1) Nos quedamos con las columnas necesarias (features + targets + date)
#    (esto evita arrastrar columnas que no usaremos)
keep_cols = ["date"] + technical_indicators_finals + [c for c in final_targets if c in df.columns]
df = df.loc[:, keep_cols].copy()

# 2) Creamos columnas *_raw explícitas (para comparar raw vs normalizado)
for c in technical_indicators_finals:
    df[f"{c}_raw"] = df[c]

# 3) Creamos columnas normalizadas *_z_exp (expanding z-score por día)
df = add_expanding_zscore_by_day(
    df,
    cols=technical_indicators_finals,
    date_col="date",
    min_periods=10,  # Ajustable: 5–15 suele ser razonable intradía
)

# 4) Armamos dataset final (raw + z_exp + targets)
final_features = (
    [f"{c}_raw" for c in technical_indicators_finals] +
    [f"{c}_z_exp" for c in technical_indicators_finals]
)

final_cols = ["date"] + final_features + [c for c in final_targets if c in df.columns]

# 5) Eliminamos filas donde aún no hay z-score (primeros min_periods-1 minutos de cada día)
mnq_features_targets_norm = df.loc[:, final_cols].dropna()

#print(mnq_features_targets_norm.head(3))


In [ ]:
mnq_features_targets_norm

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# IC (Information Coefficient) IS vs OOS
# ------------------------------------------------------------
# - IC = correlación (Spearman por defecto) entre feature y target
# - In-sample (IS): rango de fechas para entrenamiento
# - Out-of-sample (OOS): rango posterior para evaluación
# - Devuelve un DataFrame con IC_IS, IC_OOS y delta (OOS-IS)
# ============================================================

def compute_ic_is_oos(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    date_col: str = "date",
    is_end_date: str = "2023-10-26",     # Ajuste a tu corte Train/Valid, ejemplo
    oos_start_date: str = "2023-10-27",  # Ajuste a tu inicio OOS, ejemplo
    method: str = "spearman",           # 'spearman' recomendado; 'pearson' si quieres lineal
    min_obs: int = 200,                 # mínimos registros para calcular IC confiable
) -> pd.DataFrame:
    """
    Calcula IC (feature vs target) por separado en In-Sample (IS) y Out-of-Sample (OOS).

    Parámetros:
    - df: DataFrame con index datetime y columnas features/target + date_col
    - features: lista de columnas feature a evaluar (ej: raw y z_exp)
    - target: columna objetivo (ej: 'delta_pts_60' o 'delta_pts_90')
    - date_col: columna con fecha diaria (tipo date o string 'YYYY-MM-DD')
    - is_end_date: última fecha incluida en IS (YYYY-MM-DD)
    - oos_start_date: primera fecha incluida en OOS (YYYY-MM-DD)
    - method: 'spearman' (robusto a outliers y no-lineal) o 'pearson'
    - min_obs: mínimo de filas requeridas para computar IC por bloque

    Retorna:
    - DataFrame con columnas:
        feature, n_is, ic_is, n_oos, ic_oos, delta_oos_minus_is
    """

    # Copia ligera
    data = df.copy()

    # Asegura que date_col sea comparable (datetime.date)
    if date_col not in data.columns:
        raise KeyError(f"Falta columna '{date_col}' en el DataFrame.")

    # Convierte a datetime (solo fecha) para filtrar por rango
    date_series = pd.to_datetime(data[date_col]).dt.date
    is_end = pd.to_datetime(is_end_date).date()
    oos_start = pd.to_datetime(oos_start_date).date()

    # Máscara IS/OOS (no se superponen)
    mask_is = date_series <= is_end
    mask_oos = date_series >= oos_start

    # Subsets
    df_is = data.loc[mask_is, :]
    df_oos = data.loc[mask_oos, :]

    if target not in data.columns:
        raise KeyError(f"Target '{target}' no existe en el DataFrame.")

    rows = []
    for f in features:
        if f not in data.columns:
            raise KeyError(f"Feature '{f}' no existe en el DataFrame.")

        # --- IS ---
        tmp_is = df_is[[f, target]].dropna()
        n_is = len(tmp_is)
        ic_is = np.nan
        if n_is >= min_obs:
            ic_is = tmp_is[f].corr(tmp_is[target], method=method)

        # --- OOS ---
        tmp_oos = df_oos[[f, target]].dropna()
        n_oos = len(tmp_oos)
        ic_oos = np.nan
        if n_oos >= min_obs:
            ic_oos = tmp_oos[f].corr(tmp_oos[target], method=method)

        rows.append({
            "feature": f,
            "n_is": n_is,
            "ic_is": ic_is,
            "n_oos": n_oos,
            "ic_oos": ic_oos,
            "delta_oos_minus_is": (ic_oos - ic_is) if (pd.notna(ic_oos) and pd.notna(ic_is)) else np.nan
        })

    out = pd.DataFrame(rows).sort_values(by="ic_oos", ascending=False).reset_index(drop=True)
    return out





In [ ]:
IN_SPLIT_ARTIFACT = Path(os.environ.get("IN_SPLIT_ARTIFACT", "reports/stage_05_time_aware_data_splitting_summary.json"))
IN_SPLIT_ARTIFACT = DRIVE_DIR / IN_SPLIT_ARTIFACT
# Carga del JSON
with IN_SPLIT_ARTIFACT.open("r") as f:
    stage_05_time_aware_data_splitting_summary = json.load(f)

In [ ]:
from datetime import datetime

def extract_operational_dates(summary: dict) -> dict:
    splits = summary["details"]["splits"]

    def to_ymd(dt_str: str) -> str:
        return datetime.fromisoformat(dt_str).date().isoformat()

    return {
        "is_end_date": to_ymd(splits["train"]["datetime_max"]),
        "oos_start_date": to_ymd(splits["valid"]["datetime_min"]),
        "oos_end_date": to_ymd(splits["test"]["datetime_max"]),
    }

In [ ]:
dates = extract_operational_dates(stage_05_time_aware_data_splitting_summary)
#dates['is_end_date']
#dates['oos_start_date']

In [ ]:
features_to_test = [
    "price_ema60_raw", "price_ema60_z_exp",
    "momentum_10_raw", "momentum_10_z_exp",
    "roc_30_raw",      "roc_30_z_exp",
    "roc_60_raw",      "roc_60_z_exp",
 ]

In [ ]:
final_targets

In [ ]:
ic_60 = compute_ic_is_oos(
     df=mnq_features_targets_norm,
     features=features_to_test,
     target=final_targets[0],
     is_end_date=dates['is_end_date'],
     oos_start_date=dates['oos_start_date'],
     method="spearman",
     min_obs=200,
 )


In [ ]:
ic_90 = compute_ic_is_oos(
     df=mnq_features_targets_norm,
     features=features_to_test,
     target=final_targets[1],
     is_end_date=dates['is_end_date'],
     oos_start_date=dates['oos_start_date'],
     method="spearman",
     min_obs=200,
 )

In [ ]:
ic_60

In [ ]:
ic_90

#### **8.1.1. Observaciones y conclusiones**

1. Las versiones normalizadas (*_z_exp) no generalizan

    En ambos horizontes (H = 60 y H = 90):

    - Todas las features *_z_exp presentan:
      - IC positivo in-sample
      - IC cercano a cero o negativo out-of-sample
      - Un delta_oos_minus_is marcadamente negativo

    Conclusión: La normalización intradía mediante expanding z-score degradó la señal, en lugar de estabilizarla.


2. Las señales informativas están en los valores crudos

    Para ambos horizontes, las siguientes features:

    - `price_ema60_raw`  
    - `roc_60_raw`
    - `roc_30_raw`  
    - `momentum_10_raw`

    muestran:

    - IC positivo  
    - IC estable de in-sample a out-of-sample  
    - En H = 60, incluso mejora del IC OOS en algunos casos  

    Este es el comportamiento esperado de un factor robusto.


3. El horizonte H = 60 es más favorable que H = 90

    H = 60:
    - Varios factores mantienen o incrementan su IC out-of-sample  

    H = 90:
    - Todos los factores pierden algo de fuerza  
    - Aun así, mantienen IC positivo  

    Esto es consistente con un mercado intradiario de memoria corta, donde la señal se diluye a horizontes más largos.


4. Orden de importancia consistente entre horizontes

    Ranking por IC out-of-sample:

    H = 60:
    - `price_ema60_raw`
    - `roc_60_raw`  
    - `roc_30_raw`  
    - `momentum_10_raw`

    H = 90:
    - `price_ema60_raw`  
    - `roc_60_raw`  
    - `roc_30_raw`  
    - `momentum_10_raw`  

    Un ranking estable entre horizontes es señal de robustez estructural.

5. Decisión operativa

    - Eliminar las features *_z_exp  
    - Mantener únicamente las versiones raw  

**Diagnóstico final**

  Ya habíamos identificado los factores correctos.  
  Para la estructura intradía del MNQ, esta normalización no aporta valor adicional.

  Este resultado es positivo: evita complejidad innecesaria y confirma la solidez del proceso de selección previo.

### **8.2. Introducir interacciones mínimas**


No buscamos nuevos indicadores, sino capturar relaciones no lineales simples entre los cuatro factores ya validados, para que el modelo tenga acceso a información que no está en cada variable por separado.

Estas interacciones:
- no invalidan el análisis previo de IC,
- no introducen data snooping,
- suelen ser bien aprovechadas por modelos lineales y no lineales

Estas no son nuevos factores, son composiciones.

### **8.2.1. Interacciones recomendadas (mínimas y justificadas)**

#### 1. Diferencia de ROC: aceleración del movimiento

`roc_accel = roc_30 - roc_60`

Qué captura:
- Si el movimiento reciente es más fuerte que el de fondo → aceleración
- Si es más débil → desaceleración / agotamiento

Intuición
- No es tendencia (eso ya lo da EMA)
- Es cambio en la intensidad del momentum


#### 2. Momentum ponderado por estructura

`momentum_struct = momentum_10 × price_ema60_raw`

Qué captura
- Momentum alineado con la estructura dominante
- Penaliza momentum débil o contra-tendencia

Intuición
- El momentum “vale más” cuando ocurre dentro de una estructura clara

#### 3. Dirección x magnitud (desacople signo / tamaño)

`roc_signed = sign(momentum_10) × |roc_30|`

Qué captura
- Separación explícita entre:
- dirección (momentum)
- fuerza (ROC)

Intuición
- Dos movimientos con mismo ROC no son iguales si el momentum cambia de signo

### **8.2.2. Implementación**

In [ ]:
df = mnq_features_targets.copy()

# 1) Aceleración del movimiento
df["roc_accel"] = df["roc_30"] - df["roc_60"]

# 2) Momentum ponderado por estructura
df["momentum_struct"] = df["momentum_10"] * df["price_ema60"]

# 3) Dirección × magnitud
df["roc_signed"] = np.sign(df["momentum_10"]) * np.abs(df["roc_30"])

In [ ]:
interaction_features = [
    "roc_accel",
    "momentum_struct",
    "roc_signed",
]

In [ ]:
ic_60_interactions = compute_ic_is_oos(
    df=df,
    features=interaction_features,
    target=final_targets[0],
    is_end_date=dates['is_end_date'],
    oos_start_date=dates['oos_start_date'],
    method="spearman",
    min_obs=200,
)

In [ ]:
ic_90_interactions = compute_ic_is_oos(
    df=df,
    features=interaction_features,
    target=final_targets[1],
    is_end_date=dates['is_end_date'],
    oos_start_date=dates['oos_start_date'],
    method="spearman",
    min_obs=200,
)



### **8.2.3. Conclusiones — Análisis de IC para interacciones (IS vs OOS)**

In [ ]:
ic_60_interactions

In [ ]:
ic_90_interactions

1. `momentum_struct` **es válido y aporta señal**

    - **H = 60**
      - IC IS = 0.0065  
      - **IC OOS = 0.0160**  
      - Δ OOS-IS = **+0.0095**

    - **H = 90**
      - **IC OOS = 0.0091**

    **Interpretación**

    - La interacción *momentum x estructura*:
      - **mejora claramente out-of-sample**
      - generaliza mejor que varios factores base
    - Señal **real**, no atribuible a ruido

    Decisión: **Se mantiene**

2. `roc_signed` es **marginal / débil**

    - **H = 60**
      - IC OOS = 0.0068 (positivo pero bajo)
      - Δ OO-IS negativo

    - **H = 90**
      - IC OOS ≈ 0 (0.00026)
      - Pérdida marcada de señal OOS

    **Interpretación**

    - No es dañino, pero:
      - no agrega valor consistente
      - la señal se diluye al aumentar el horizonte

    Decisión: **Descartable**, o dejar solo para pruebas exploratorias

3. `roc_accel` **no aporta señal**

    - IC **negativo** tanto in-sample como out-of-sample  
    - No se observa relación explotable con el target

    Decisión: **Descartar sin dudar**

4. Comparación contra factor base  `momentum_struct` vs `momentum_10`


In [ ]:
momentum_features = [
    "momentum_10",
    "momentum_struct",
    ]

ic_60_momentum = compute_ic_is_oos(
    df=df,
    features=momentum_features,
    target=final_targets[0],
    is_end_date=dates['is_end_date'],
    oos_start_date=dates['oos_start_date'],
    method="spearman",
    min_obs=200,
)

ic_90_momentum = compute_ic_is_oos(
    df=df,
    features=momentum_features,
    target=final_targets[1],
    is_end_date=dates['is_end_date'],
    oos_start_date=dates['oos_start_date'],
    method="spearman",
    min_obs=200,
)

In [ ]:
ic_60_momentum

In [ ]:
ic_90_momentum

- **`momentum_struct`**
  - Generaliza mejor que `momentum_10`
  - IC OOS **alto y estable** en **H=60** y **H=90**
  - Señal robusta fuera de muestra

- **`momentum_10`**
  - IC OOS menor e **inestable**
  - Se degrada claramente al pasar de **H=60 → H=90**

En conclusión:
- El **momentum alineado con estructura** es superior al momentum puro.
- **Decisión**: usar `momentum_struct` como feature principal; `momentum_10` es prescindible.

5. Decisión final de features

**Features base**
- `price_ema60`
- `roc_60`
- `roc_30`
- `momentum_10` (Se mantiene para calcular `momentum_struct`)

**Interacción aceptada**
- `momentum_struct`

**Features descartadas**

- `*_z_exp`
- `roc_signed`
- `roc_accel`



In [ ]:
# 2) Momentum ponderado por estructura
df["momentum_struct"] = df["momentum_10"] * df["price_ema60"]

Diagnóstico final

> La señal intradía del MNQ **vive en los valores crudos**  
> y **se potencia cuando el momentum está alineado con la estructura**

### **8.3. Verificar estabilidad por horizonte**


#### **8.3.1. Introducción**


En predicción intradía, **un mismo indicador no necesariamente funciona igual para distintos horizontes de predicción**.  
Un feature puede capturar dinámicas de corto plazo (por ejemplo, impulsos rápidos), pero perder relevancia cuando el horizonte se extiende, o viceversa.

Por este motivo, no basta con que un indicador tenga **IC positivo**:  es necesario evaluar **si su relación con el target es estable cuando cambia el horizonte** \(H\).

Este análisis permite distinguir entre:
- señales **estructurales** del mercado, y
- señales **dependientes del horizonte** o directamente inestables.

El objetivo es que para cada feature, se busca determinar si:

- **Generaliza entre horizontes** (H = 60 y H = 90),
- Es **específica de un horizonte**, o
- Es **inestable** y debe descartarse.

En esta etapa **no se maximiza el IC**, sino que se evalúa **consistencia out-of-sample**.

**Criterios de clasificación**

Para cada feature \(f\):

1. Feature **estable**
    - IC OOS positivo en **H = 60 y H = 90**
    - Mantiene el signo
    - La variación de magnitud al aumentar H es moderada y explicable
    > Puede usarse en ambos horizontes.

2. Feature **especializada**
    - IC OOS positivo solo en **un horizonte**
    - IC OOS débil o cercano a cero en el otro
    - Sin comportamiento errático

    > Se utiliza únicamente en el horizonte donde aporta señal.

3. Feature **inestable**
    - IC OOS negativo
    - Cambios de signo sin patrón
    - Ruptura clara al variar el horizonte

    > Se descarta.

**Intuición clave**

- Un feature **estable** refleja una dinámica persistente del mercado.
- Un feature **especializado** captura efectos de escala temporal específica.
- Un feature **inestable** suele ser ruido o sobreajuste.

Este enfoque evita forzar indicadores fuera de su dominio natural de validez.

**Resultado esperado**

- Clasificar los features en lugar de eliminarlos arbitrariamente.
- Definir un **set coherente por horizonte**.
- Preparar el terreno para modelos que generalicen mejor out-of-sample.


#### **8.3.2. Implementación**


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Estabilidad por horizonte (H=60 vs H=90) usando IC IS/OOS
# ------------------------------------------------------------
# Requiere:
# - mnq_features_targets_norm: DataFrame con columnas features y targets
# - compute_ic_is_oos(df, features, target, ...) ya definida
# ============================================================

def evaluate_feature_stability_by_horizon(
    df: pd.DataFrame,
    features: list[str],
    target_h60: str = final_targets[0],
    target_h90: str = final_targets[1],
    date_col: str = "date",
    is_end_date: str = dates['is_end_date'],
    oos_start_date: str = dates['oos_start_date'],
    method: str = "spearman",
    min_obs: int = 200,
    eps: float = 1e-12,
) -> pd.DataFrame:
    """
    Calcula IC IS/OOS para H=60 y H=90 y clasifica cada feature como:
    - stable:    IC OOS > 0 en ambos horizontes y signos consistentes
    - specialized_h60: aporta en H=60 pero no en H=90
    - specialized_h90: aporta en H=90 pero no en H=60
    - unstable:  IC OOS <= 0 en ambos o comportamiento errático

    Retorna un DataFrame con:
    - ic_is_60, ic_oos_60, delta_60
    - ic_is_90, ic_oos_90, delta_90
    - ratio_oos_90_over_60 (magnitud relativa)
    - stability_label
    """
    # IC por horizonte
    ic60 = compute_ic_is_oos(
        df=df,
        features=features,
        target=target_h60,
        date_col=date_col,
        is_end_date=is_end_date,
        oos_start_date=oos_start_date,
        method=method,
        min_obs=min_obs,
    ).rename(columns={
        "ic_is": "ic_is_60",
        "ic_oos": "ic_oos_60",
        "delta_oos_minus_is": "delta_60",
        "n_is": "n_is_60",
        "n_oos": "n_oos_60",
    })

    ic90 = compute_ic_is_oos(
        df=df,
        features=features,
        target=target_h90,
        date_col=date_col,
        is_end_date=is_end_date,
        oos_start_date=oos_start_date,
        method=method,
        min_obs=min_obs,
    ).rename(columns={
        "ic_is": "ic_is_90",
        "ic_oos": "ic_oos_90",
        "delta_oos_minus_is": "delta_90",
        "n_is": "n_is_90",
        "n_oos": "n_oos_90",
    })

    # Merge por feature
    out = ic60.merge(ic90, on="feature", how="inner")

    # Razón de magnitudes (cuánto queda de la señal al pasar de 60 -> 90)
    out["ratio_oos_90_over_60"] = (
        out["ic_oos_90"].abs() / (out["ic_oos_60"].abs() + eps)
    )

    # Etiquetado de estabilidad por horizonte
    def _label(row) -> str:
        o60 = row["ic_oos_60"]
        o90 = row["ic_oos_90"]

        # Si falta info OOS en alguno, marcar como unknown
        if pd.isna(o60) or pd.isna(o90):
            return "unknown"

        pos60 = o60 > 0
        pos90 = o90 > 0

        # Estable: positivo en ambos horizontes
        if pos60 and pos90:
            return "stable"

        # Especializado: solo uno aporta
        if pos60 and not pos90:
            return "specialized_h60"
        if pos90 and not pos60:
            return "specialized_h90"

        # Inestable: negativo o ~0 en ambos (según criterio de signo)
        return "unstable"

    out["stability_label"] = out.apply(_label, axis=1)

    # Orden sugerido: primero lo estable (y por IC OOS promedio)
    out["ic_oos_mean"] = out[["ic_oos_60", "ic_oos_90"]].mean(axis=1)
    out = out.sort_values(
        by=["stability_label", "ic_oos_mean"],
        ascending=[True, False],
    ).reset_index(drop=True)

    # Columnas finales (ordenadas)
    cols = [
        "feature",
        "stability_label",
        "ic_is_60", "ic_oos_60", "delta_60",
        "ic_is_90", "ic_oos_90", "delta_90",
        "ratio_oos_90_over_60",
        "n_is_60", "n_oos_60",
        "n_is_90", "n_oos_90",
    ]
    return out[cols]


# ============================================================
# EJEMPLO DE USO
# ============================================================
# Lista de features a evaluar (ajustá a tu set final)
# Ejemplo: factores base + interacción validada
features_to_check = [
    "price_ema60",
    "roc_60",
    "roc_30",
    "momentum_10",
    "momentum_struct",
]

stability_report = evaluate_feature_stability_by_horizon(
    df=df,
    features=features_to_check,
    target_h60="delta_pts_60",
    target_h90="delta_pts_90",
    date_col="date",
    is_end_date="2023-10-26",
    oos_start_date="2023-10-27",
    method="spearman",
    min_obs=200,
)

#### **8.3.3.  Conclusiones — Estabilidad por horizonte y decisión sobre momentum**


In [ ]:
stability_report

1. Todos los features resultan **estables** (criterio formal)

    - IC OOS **positivo en H = 60 y H = 90**
    - Sin cambio de signo
    - No se detectan features inestables

      > El set es **coherente y consistente entre horizontes**.

2. Las diferencias aparecen en la **calidad de la estabilidad**

    No todos los features estables aportan el mismo valor.

    **Más robustos (mejor balance H=60 → H=90):**
    - `price_ema60`
    - `roc_60`

    Ambos mantienen:
    - IC OOS relativamente alto
    - Ratios cercanos a 1, indicando **persistencia de señal**

3. `momentum_struct` es **estable y valioso**

    - Muy fuerte en **H = 60**
    - En **H = 90**:
      - El IC OOS disminuye, pero
      - se mantiene **claramente positivo**
      - sin comportamiento errático

      > Feature estable, con **mayor relevancia en horizontes más cortos**.

4. `roc_30` y `momentum_10` son **estables pero más frágiles**

    - Caída clara del IC OOS al pasar de H = 60 a H = 90
    - Ratios bajos (≈ 0.4 – 0.7)

      > Aportan señal, pero de forma **secundaria**, especialmente en H = 90.

5. Justificación: por qué usar `momentum_struct` y no `momentum_10`

    Aunque `momentum_10` es formalmente estable, su señal:

    - es **más débil out-of-sample**,
    - se **degrada claramente** al aumentar el horizonte,
    - y queda **contenida** dentro de `momentum_struct`.

    `momentum_struct` combina:
    - dirección de corto plazo (momentum),
    - con alineación estructural (tendencia / contexto),

    lo que produce:
    - **mejor generalización OOS**,
    - mayor estabilidad entre horizontes,
    - y menor redundancia informativa.

    > Por simplicidad, robustez y poder explicativo,  
    **se utiliza `momentum_struct` como único feature de momentum**.

6. Decisión final de features

    **Core multi-horizonte**
    - `price_ema60`
    - `roc_60`

    **Refuerzo (especialmente H = 60)**
    - `momentum_struct`

    **Secundario**
    - `roc_30`

    **Descartado**
    - `momentum_10`

---

### Diagnóstico final

> El modelo intradía del MNQ se beneficia de
> **estructura + momentum alineado**, no de momentum aislado.

Con este criterio, el feature set queda **cerrado, parsimonioso y validado out-of-sample**,  listo para pasar a la etapa de **modelado**.

### **8.4. Separar señales de estructura, dirección y magnitud**

Después de seleccionar y validar features (IC IS/OOS + estabilidad por horizonte), organizamos el set final por **rol informativo**:

- **Estructura / régimen** (`price_ema60`): describe el “contexto” dominante del mercado.
- **Dirección alineada** (`momentum_struct`): captura el empuje direccional cuando está en sintonía con la estructura.
- **Magnitud / intensidad** (`roc_30`, `roc_60`): aproxima cuán grande puede ser el movimiento (escala/cambio).

**Por qué se hace:**
1. **Orden y claridad**: cada feature cumple un propósito distinto y evita ambigüedades.
2. **Mejor inductive bias**: el modelo aprende “dónde está el mercado”, “hacia dónde empuja” y “cuánto podría moverse”.
3. **Mejor interpretabilidad**: permite diagnosticar rápidamente qué tipo de señal está aportando (o no) cada grupo.

Este paso **no agrega indicadores**: solo formaliza lo que ya fue respaldado empíricamente.

In [ ]:
import pandas as pd

# ============================================================
# Validación práctica del esquema "Estructura / Dirección / Magnitud"
# ------------------------------------------------------------
# Requiere:
# - mnq_features_targets_norm (DataFrame)
# - compute_ic_is_oos(...) ya definida
# ============================================================

# 1) Definir roles (ajusta nombres si tus columnas no tienen sufijos)
ROLE_MAP = {
    "structure":  ["price_ema60"],          # o "price_ema60_raw" si tu dataset usa raw
    "direction":  ["momentum_struct"],
    "magnitude":  ["roc_30", "roc_60"],     # idem: _raw si corresponde
}

# 2) Helper: calcula IC IS/OOS por feature y agrega columna "role"
def ic_by_role(
    df: pd.DataFrame,
    role_map: dict[str, list[str]],
    target: str,
    is_end_date: str,
    oos_start_date: str,
    method: str = "spearman",
    min_obs: int = 200,
    date_col: str = "date",
) -> pd.DataFrame:
    rows = []
    for role, feats in role_map.items():
        ic_tbl = compute_ic_is_oos(
            df=df,
            features=feats,
            target=target,
            date_col=date_col,
            is_end_date=is_end_date,
            oos_start_date=oos_start_date,
            method=method,
            min_obs=min_obs,
        ).copy()
        ic_tbl.insert(0, "role", role)
        rows.append(ic_tbl)

    out = pd.concat(rows, ignore_index=True)
    # Orden: por IC OOS descendente (lo que más importa)
    out = out.sort_values(["ic_oos"], ascending=False).reset_index(drop=True)
    return out


# 3) Ejecutar para H=60 y H=90 (ajusta fechas a tu split real)
IS_END = "2023-10-26"
OOS_START = "2023-10-27"

ic_roles_60 = ic_by_role(
    df=df,
    role_map=ROLE_MAP,
    target="delta_pts_60",
    is_end_date=IS_END,
    oos_start_date=OOS_START,
)

ic_roles_90 = ic_by_role(
    df=df,
    role_map=ROLE_MAP,
    target="delta_pts_90",
    is_end_date=IS_END,
    oos_start_date=OOS_START,
)

print("IC por rol (H=60):")
display(ic_roles_60)

print("\nIC por rol (H=90):")
display(ic_roles_90)


# 4) Validación adicional (simple): resumen por rol (promedios OOS)
def summarize_role_strength(ic_table: pd.DataFrame) -> pd.DataFrame:
    return (
        ic_table
        .groupby("role", as_index=False)
        .agg(
            n_features=("feature", "count"),
            ic_oos_mean=("ic_oos", "mean"),
            ic_oos_max=("ic_oos", "max"),
            ic_oos_min=("ic_oos", "min"),
        )
        .sort_values("ic_oos_mean", ascending=False)
        .reset_index(drop=True)
    )

role_summary_60 = summarize_role_strength(ic_roles_60)
role_summary_90 = summarize_role_strength(ic_roles_90)

print("\nResumen por rol (H=60):")
display(role_summary_60)

print("\nResumen por rol (H=90):")
display(role_summary_90)


**H = 60:**
  
  - Estructura > Dirección > Magnitud
  - El momentum alineado (momentum_struct) es casi tan relevante como la estructura.

**H = 90:**

- Estructura ≈ Magnitud > Dirección
- La señal direccional pierde peso; dominan estructura y escala.

**Lectura final:**

A corto plazo manda la dirección alineada con estructura; al extender el horizonte, prevalece el contexto estructural y la magnitud del movimiento.

Este punto queda correctamente alineado con el Capítulo 4 (Feature Engineering / Alpha Factors).

Por qué:
- Seleccionó factores con IC IS/OOS y control de correlación.
- Validó generalización por horizonte.
- Refinó interacciones mínimas con respaldo OOS.
- Organizó el set por rol informativo (estructura, dirección, magnitud).

Cerró un feature set parsimonioso y robusto, listo para modelado.

Conclusión:

La notebook de feature engineering ya refleja fielmente el enfoque metodológico del capítulo: menos factores, mejor justificados, validados fuera de muestra.
